# 🎲 Générer des données synthétiques pour la classification

Notebook **Cheat-sheet + Tutoriel** (avec un peu de **Wiki**) : produire des **datasets synthétiques** pour la classification (et la régression), les **visualiser**, et comprendre *quel générateur choisir pour quelle frontière de décision*.

Pourquoi générer des données synthétiques ?

- **Tester / débugger** un algorithme sans dépendre d'une vraie donnée.
- **Illustrer** un concept (non-linéarité, déséquilibre, recouvrement de classes).
- **Contrôler** précisément la difficulté (séparabilité, bruit, nombre de classes).
- **Augmenter** ou **anonymiser** (synthetic data qui préserve les corrélations).

Plan :

1. `make_gaussian_quantiles` — classes par quantiles d'une gaussienne.
2. `make_circles` — cercles concentriques (non-linéaire radial).
3. `make_moons` — deux demi-lunes (non-linéaire).
4. `make_blobs` — clusters gaussiens convexes.
5. `make_classification` (+ `make_multilabel_classification`) — le générateur paramétrable.
6. Panorama 2D comparatif.
7. `make_regression` — cible continue + vrais coefficients.
8. Déséquilibre contrôlé pour le benchmark.
9. Construire un **dataset composite** large from scratch.
10. Export CSV reproductible.
11. `drawdata` — données dessinées à la main.
12. Données mixtes (num + cat) avec **Faker**.
13. **SDV** & modèles génératifs (panorama 2026).

## 0. Setup — imports & charte graphique

On fixe une **graine globale** (`random` + `numpy`) pour la reproductibilité (consigne projet : `seed=42`), et on importe la pile scientifique standard.

In [ ]:
import math
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

On définit une **charte de couleurs** unique (8 teintes sémantiques) et deux helpers réutilisables :

- `apply_chart_style()` — configure seaborn/matplotlib une fois pour toutes.
- `scatter_by_category()` — scatter 2D coloriant une variable catégorielle par **N couleurs distinctes** de la palette (règle : catégories sans ordre naturel → couleurs distinctes prises dans l'ordre, pas de panachage).

In [ ]:
CHART: dict[str, str] = {
    "primary_1":   "#00798c",  # Teal     — info / catégorie principale
    "mauvais":     "#d1495b",  # Crimson  — bad / critique
    "moyen":       "#edae49",  # Saffron  — moyen / warning
    "accent":      "#66a182",  # Sage     — accent / bon
    "accent_dark": "#2e4057",  # Navy     — texte fort
    "lavender":    "#9d83b8",  # Violet               — secondaire 1
    "dusty_rose":  "#b8848e",  # Rose terne           — secondaire 2
    "beige":       "#c9b78b",  # Beige                — neutre
}
PALETTE: list[str] = list(CHART.values())


def apply_chart_style() -> None:
    """Applique la charte graphique au runtime matplotlib + seaborn (idempotent)."""
    sns.set_theme(style="whitegrid", palette=PALETTE)
    plt.rcParams.update({
        "figure.figsize": (6, 6),
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.edgecolor": CHART["accent_dark"],
        "axes.labelcolor": CHART["accent_dark"],
        "axes.titleweight": "bold",
        "grid.alpha": 0.3,
        "font.size": 11,
    })


def scatter_by_category(
    df: pd.DataFrame,
    x: str,
    y: str,
    hue: str,
    ax: "plt.Axes | None" = None,
    title: str | None = None,
    s: int = 18,
) -> "plt.Axes":
    """Scatter 2D coloriant `hue` (catégoriel) par N couleurs distinctes de la PALETTE.

    Args:
        df: données.
        x, y: colonnes des axes.
        hue: colonne catégorielle encodée par la couleur.
        ax: axe matplotlib cible (créé si None).
        title: titre optionnel.
        s: taille des points.
    Returns:
        L'axe matplotlib utilisé.
    """
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 6))
    levels = sorted(pd.unique(df[hue]))
    colors = [PALETTE[i % len(PALETTE)] for i in range(len(levels))]
    sns.scatterplot(data=df, x=x, y=y, hue=hue, hue_order=levels,
                    palette=colors, s=s, edgecolor="white", linewidth=0.2, ax=ax)
    if title:
        ax.set_title(title)
    return ax


apply_chart_style()

La charte est maintenant active : toutes les figures du notebook partagent le même style.

## 1. make_gaussian_quantiles

`make_gaussian_quantiles` tire des points d'une **gaussienne multivariée** $\mathcal{N}(\mu, \Sigma)$, puis découpe en classes selon les **quantiles de la distance de Mahalanobis** au centre. Pour `n_classes=k`, chaque classe est un anneau concentrique contenant $1/k$ des points :

$$ d(x) = \sqrt{(x-\mu)^\top \Sigma^{-1} (x-\mu)} \quad\text{puis classes} = \text{quantiles}(d) $$

Plus `cov` est grand, plus la gaussienne est étalée. Utile pour des frontières **radiales** difficiles.

In [ ]:
from sklearn.datasets import make_gaussian_quantiles

# Une gaussienne unique découpée en 2 classes par quantiles de distance au centre
X, y = make_gaussian_quantiles(cov=3.0, n_samples=10_000, n_features=2,
                               n_classes=2, random_state=1)
df = pd.DataFrame(X, columns=["x1", "x2"])
df["y"] = y

scatter_by_category(df, "x1", "x2", "y",
                    title="make_gaussian_quantiles — 2 classes par quantile")
plt.show()

On peut **composer** deux gaussiennes décalées pour fabriquer des classes imbriquées : on concatène le premier jeu avec un second centré en `(4, 4)`, dont on inverse les labels.

In [ ]:
# Composition : 2ᵉ gaussienne décentrée (mean=(4,4)) → classes imbriquées
X2, y2 = make_gaussian_quantiles(mean=(4, 4), cov=1.0, n_samples=5_000,
                                 n_features=2, n_classes=2, random_state=1)
temp = pd.DataFrame(X2, columns=["x1", "x2"])
temp["y"] = 1 - y2  # inverse les labels pour imbriquer les classes
result = pd.concat([df, temp], ignore_index=True)

scatter_by_category(result, "x1", "x2", "y",
                    title="Deux gaussian_quantiles concaténées")
plt.show()

## 2. make_circles

Deux **cercles concentriques** : un classifieur linéaire échoue (frontière radiale, pas une droite). Idéal pour montrer l'intérêt des noyaux (SVM RBF), de KNN ou d'un réseau de neurones.

- `factor` ∈ ]0, 1[ : ratio rayon intérieur / extérieur.
- `noise` : écart-type du bruit gaussien ajouté.

In [ ]:
from sklearn.datasets import make_circles

X, y = make_circles(n_samples=1_000, factor=0.5, noise=0.05, random_state=17)
df = pd.DataFrame(X, columns=["x1", "x2"])
df["y"] = y

scatter_by_category(df, "x1", "x2", "y",
                    title="make_circles — cercles concentriques (non-linéaire)")
plt.show()

## 3. make_moons

Deux **demi-lunes** entrelacées : le cas non-linéaire le plus classique pour visualiser une frontière de décision courbe. `noise` contrôle le chevauchement des deux classes.

In [ ]:
from sklearn.datasets import make_moons

X, y = make_moons(n_samples=10_000, noise=0.12, random_state=RANDOM_STATE)
df = pd.DataFrame(X, columns=["x1", "x2"])
df["y"] = y

scatter_by_category(df, "x1", "x2", "y",
                    title="make_moons — 2 demi-lunes (non-linéaire)")
plt.show()

## 4. make_blobs

Des **clusters gaussiens isotropes** (convexes) : le cas « facile », linéairement séparable si les centres sont assez écartés. Référence pour tester KMeans, GMM, ou un classifieur de base. `cluster_std` règle la dispersion (donc le recouvrement).

In [ ]:
from sklearn.datasets import make_blobs

X, y = make_blobs(n_samples=10_000, centers=3, n_features=2,
                  cluster_std=2.0, random_state=1)
df = pd.DataFrame(X, columns=["x1", "x2"])
df["y"] = y

scatter_by_category(df, "x1", "x2", "y",
                    title="make_blobs — 3 clusters gaussiens convexes")
plt.show()

## 5. make_classification

Le générateur **le plus paramétrable** : il place les classes sur les sommets d'un hypercube puis ajoute features redondantes / répétées / bruit. Paramètres clés :

| Paramètre | Rôle |
|---|---|
| `n_samples` | nombre d'observations |
| `n_features` | nombre total de features |
| `n_informative` | features qui portent réellement l'info |
| `n_redundant` | combinaisons linéaires des informatives |
| `n_repeated` | duplications exactes |
| `n_classes` | nombre de classes |
| `n_clusters_per_class` | sous-clusters par classe |
| `weights` | déséquilibre des classes, ex. `[0.2, 0.8]` |
| `flip_y` | fraction de labels bruités (bruit cible) |
| `class_sep` | séparabilité (plus grand = plus facile) |

Ci-dessous un cas **déséquilibré 20/80** avec une bonne séparation (`class_sep=1.5`) et 1 % de bruit cible.

In [ ]:
from sklearn.datasets import make_classification

X, y = make_classification(n_samples=10_000, n_features=2, n_informative=2,
                           n_redundant=0, n_repeated=0, n_classes=2,
                           n_clusters_per_class=1, class_sep=1.5,
                           flip_y=0.01, weights=[0.2, 0.8], random_state=3)
df = pd.DataFrame(X, columns=["x1", "x2"])
df["y"] = y

scatter_by_category(df, "x1", "x2", "y",
                    title="make_classification — déséquilibré 20/80, class_sep=1.5")
plt.show()

### 5.1 make_multilabel_classification

En **multi-label**, une observation peut appartenir à *plusieurs* classes simultanément. `Y` est alors une matrice binaire `(n_samples, n_classes)` (et non un vecteur). Modèle génératif : chaque échantillon est un sac de mots tiré de plusieurs « sujets ».

In [ ]:
from sklearn.datasets import make_multilabel_classification

X, Y = make_multilabel_classification(n_samples=1_000, n_features=7, n_classes=3,
                                      n_labels=2, random_state=1)
print("Shapes — X:", X.shape, "| Y (matrice binaire):", Y.shape)
for i in range(5):
    print("features:", X[i].astype(int), "-> labels:", Y[i])

## 6. Panorama 2D comparatif

Tous les générateurs côte à côte : un coup d'œil pour choisir selon la **forme de frontière** voulue (convexe vs non-linéaire vs radiale vs déséquilibrée).

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
generators = [
    ("make_blobs(4)", lambda: make_blobs(n_samples=300, centers=4,
                                         cluster_std=1.0, random_state=RANDOM_STATE)),
    ("make_moons", lambda: make_moons(n_samples=300, noise=0.15,
                                      random_state=RANDOM_STATE)),
    ("make_circles", lambda: make_circles(n_samples=300, factor=0.5,
                                          noise=0.1, random_state=RANDOM_STATE)),
    ("gaussian_quantiles(3)", lambda: make_gaussian_quantiles(
        n_samples=300, n_classes=3, random_state=RANDOM_STATE)),
    ("make_classification", lambda: make_classification(
        n_samples=300, n_features=2, n_redundant=0,
        n_clusters_per_class=2, random_state=RANDOM_STATE)),
    ("classification 90/10", lambda: make_classification(
        n_samples=300, n_features=2, n_redundant=0,
        weights=[0.9, 0.1], random_state=RANDOM_STATE)),
]
for ax, (name, gen) in zip(axes.ravel(), generators):
    Xg, yg = gen()
    dfg = pd.DataFrame(Xg, columns=["x1", "x2"])
    dfg["y"] = yg
    scatter_by_category(dfg, "x1", "x2", "y", ax=ax, title=name, s=12)
    ax.legend([], frameon=False)
plt.tight_layout()
plt.show()

## 7. make_regression

Pour la **régression**, `make_regression` génère $y = X\beta + \varepsilon$ avec un bruit gaussien `noise`. Avec `coef=True`, on récupère les **vrais coefficients** $\beta$ — précieux pour valider un algo de sélection de variables (les `n_informative` coefs non nuls vs le reste à zéro).

In [ ]:
from sklearn.datasets import make_regression

X, y, coef = make_regression(n_samples=500, n_features=10, n_informative=5,
                             noise=10.0, coef=True, random_state=RANDOM_STATE)
print("X:", X.shape, "| coefs non nuls (informatifs):", int((coef != 0).sum()))
print("Vrais coefficients beta:", np.round(coef, 1))

## 8. Déséquilibre contrôlé

Un helper réutilisable pour fabriquer un dataset binaire avec un **ratio de positifs** choisi (ex. 5 %). Utile pour tester les stratégies anti-déséquilibre (class weights, SMOTE — cf. `NLP_Classification_Smote`).

In [ ]:
def imbalance_dataset(n: int = 1_000, ratio: float = 0.05, n_features: int = 10,
                      random_state: int = RANDOM_STATE) -> tuple[np.ndarray, np.ndarray]:
    """Dataset binaire avec `ratio` de positifs (classe 1).

    Args:
        n: nombre d'observations.
        ratio: proportion de la classe minoritaire (1).
        n_features: nombre de variables.
        random_state: graine.
    Returns:
        (X, y) — features et cible binaire.
    """
    X, y = make_classification(
        n_samples=n, n_features=n_features, n_informative=n_features // 2,
        n_classes=2, weights=[1 - ratio, ratio], flip_y=0.01,
        random_state=random_state,
    )
    return X, y


X, y = imbalance_dataset(n=2_000, ratio=0.05)
print(f"Positifs : {y.sum()} / {len(y)} ({y.mean():.1%})")

## 9. Construire un dataset composite from scratch

Au-delà des générateurs prêts à l'emploi, on peut **assembler** plusieurs structures pour fabriquer un dataset large et difficile : plusieurs blocs de features (lunes, anneaux gaussiens, gaussiennes) **juxtaposés en colonnes**, plus des features de **bruit pur**. C'est un excellent banc d'essai pour la **sélection de variables** (séparer le signal du bruit).

**Bloc 1 — Moons + anneau** : deux demi-lunes (classes 0/1) auxquelles on ajoute un anneau de classe 2 autour du centroïde (rayon bruité).

In [ ]:
N_ROWS = 15_000

# Bloc Moons : 2 demi-lunes (classes 0/1) + un anneau de classe 2 autour du centroïde
Xm, ym = make_moons(n_samples=10_000, noise=0.2, random_state=RANDOM_STATE)
centroide = (Xm[:, 0].mean(), Xm[:, 1].mean())
ring_sigma = 0.2
r = max(Xm.min(), Xm.max(), key=abs) - max(centroide, key=abs)
ring_pts = []
for _ in range(5_000):
    R = r + random.gauss(0, ring_sigma)
    theta = random.uniform(0, 1) * 2 * math.pi
    ring_pts.append([(centroide[0] + R - 0.3) * math.cos(theta) + 0.5,
                     (centroide[1] + R - 0.8) * math.sin(theta) + 0.2])
Xm = np.vstack([Xm, np.array(ring_pts)])
ym = np.concatenate([ym, np.full(5_000, 2)])
df_moons = pd.DataFrame(Xm, columns=["Moons_1", "Moons_2"]).head(N_ROWS).reset_index(drop=True)

**Bloc 2 — Circles** (deux `gaussian_quantiles` 3-classes concaténées) et **Bloc 3 — Gaussians** (`make_classification` 3-classes), ce dernier portant la **cible finale** `y`.

In [ ]:
# Bloc Circles : 2 gaussian_quantiles 3-classes concaténées
X1, y1 = make_gaussian_quantiles(cov=3.0, n_samples=10_000, n_features=2,
                                 n_classes=3, random_state=1)
df_a = pd.DataFrame(X1, columns=["Circles_1", "Circles_2"])
X2, y2 = make_gaussian_quantiles(mean=(4, 4), cov=1.0, n_samples=5_000,
                                 n_features=2, n_classes=3, random_state=2)
df_b = pd.DataFrame(X2, columns=["Circles_1", "Circles_2"])
df_circles = pd.concat([df_a, df_b], ignore_index=True).head(N_ROWS).reset_index(drop=True)

# Bloc Gaussians : make_classification 3-classes — porte la CIBLE finale
Xg, yg = make_classification(n_samples=15_025, n_features=2, n_informative=2,
                             n_redundant=0, n_repeated=0, n_classes=3,
                             n_clusters_per_class=1, class_sep=0.5, flip_y=0.02,
                             weights=None, random_state=17)
df_gaussian = pd.DataFrame(Xg, columns=["Gaussians_1", "Gaussians_2"])
df_gaussian["y"] = yg
df_gaussian = df_gaussian.head(N_ROWS).reset_index(drop=True)

**Assemblage** : on juxtapose les 3 blocs en colonnes, on ajoute 14 features de **bruit gaussien pur** (sigma aléatoire par colonne), on place la cible en tête et on mélange les lignes.

In [ ]:
# Concat des blocs par COLONNES (datasets larges côte à côte)
df_classif = pd.concat(
    [df_moons, df_circles, df_gaussian.drop(columns=["y"])], axis=1
)

# 14 features de bruit gaussien pur (sigma aléatoire par colonne)
for k in range(1, 15):
    sigma = random.randint(10, 600) / 100
    df_classif[f"Random_{k}"] = np.random.normal(0, sigma, size=N_ROWS)

# Cible en tête + shuffle
df_classif.insert(0, "y", df_gaussian["y"].to_numpy())
df_classif = df_classif.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
print("Dataset composite:", df_classif.shape)
print("Répartition des classes:", df_classif["y"].value_counts().to_dict())
df_classif.head()

## 10. Export CSV

On sauvegarde le dataset dans `data/<notebook>/` (chemin **relatif** au projet, reproductible — l'original exportait vers Google Drive). Séparateur `;` pour ouverture directe en tableur FR.

In [ ]:
from pathlib import Path

DATA_DIR = Path("data/Structure_Generer_Donnees_Classification")
DATA_DIR.mkdir(parents=True, exist_ok=True)
out_path = DATA_DIR / "df_classification.csv"
df_classif.to_csv(out_path, sep=";", index=False)
print("Export ->", out_path, "| existe:", out_path.exists())

## 11. drawdata — données dessinées à la main

La librairie **[`drawdata`](https://github.com/koaning/drawdata)** affiche un widget interactif dans le notebook : on **dessine** des points à la souris, classe par classe, puis on récupère le tableau. C'est parfait pour fabriquer rapidement un cas pédagogique sur mesure.

```text
# Installation puis lancement du widget (interactif, dans Jupyter) :
%pip install drawdata
from drawdata import ScatterWidget
widget = ScatterWidget()
widget                      # on dessine, puis widget.data_as_pandas
```

Le widget étant **interactif** (il ne s'exécute pas en mode headless), on conserve ci-dessous la **donnée effectivement dessinée** (1464 points, 4 classes `a`/`b`/`c`/`d`) et on la rejoue : le graphique d'origine est ainsi préservé et reproductible.

In [ ]:
data = [{"x":293.2792877084937,"y":384.9242279742612,"color":"a"},{"x":253.49061746156286,"y":373.56719247723123,"color":"a"},{"x":280.44201707688205,"y":392.931174318797,"color":"a"},{"x":303.0484525853223,"y":396.16509712876933,"color":"a"},{"x":278.96645090077703,"y":406.51953326064285,"color":"a"},{"x":258.30056997465476,"y":373.8191989232063,"color":"a"},{"x":256.41467895334006,"y":394.1602741401908,"color":"a"},{"x":234.4062325463434,"y":403.18572736632393,"color":"a"},{"x":221.59044919651643,"y":389.3752640639944,"color":"a"},{"x":203.10657501814075,"y":370.6600899991974,"color":"a"},{"x":186.8604775373891,"y":394.7202497122611,"color":"a"},{"x":203.17719896051128,"y":428.13595792536944,"color":"a"},{"x":171.14804757246097,"y":392.78378800049626,"color":"a"},{"x":195.86370621443885,"y":359.67365140605676,"color":"a"},{"x":134.41563717275164,"y":364.64247870137564,"color":"a"},{"x":137.1185525358622,"y":378.6374716842663,"color":"a"},{"x":136.713497486844,"y":420.6933522581343,"color":"a"},{"x":92.42829707152366,"y":373.84313274836114,"color":"a"},{"x":137.59286992383704,"y":339.99794114007926,"color":"a"},{"x":107.4860592404943,"y":362.93346387852785,"color":"a"},{"x":119.25262224590575,"y":341.2824526813464,"color":"a"},{"x":50.333496138963504,"y":355.1762856458888,"color":"a"},{"x":146.27214538720887,"y":329.1278701495558,"color":"a"},{"x":90.91811624828419,"y":314.9130246684163,"color":"a"},{"x":84.31547378862601,"y":287.10675328825596,"color":"a"},{"x":42.50953678546223,"y":304.3687222866488,"color":"a"},{"x":131.3159163099218,"y":275.85165434696114,"color":"a"},{"x":48.21678497364087,"y":270.7073973474785,"color":"a"},{"x":38.291813863488066,"y":212.20541415113877,"color":"a"},{"x":112.22050889631197,"y":224.2075151641767,"color":"a"},{"x":55.89227057252316,"y":183.6359444964923,"color":"a"},{"x":127.55366813581557,"y":205.53972631305578,"color":"a"},{"x":47.63135550110023,"y":174.92782592105948,"color":"a"},{"x":118.59530320490754,"y":216.1532822221513,"color":"a"},{"x":91.40964357672269,"y":187.62081566334678,"color":"a"},{"x":111.37300336732719,"y":144.77769650658678,"color":"a"},{"x":88.95406268606482,"y":125.96892187727411,"color":"a"},{"x":96.61101636338486,"y":119.41193260003217,"color":"a"},{"x":166.93544678163184,"y":175.28702916300233,"color":"a"},{"x":127.30518711254551,"y":124.36826702853114,"color":"a"},{"x":161.7590628672451,"y":165.65149209317872,"color":"a"},{"x":197.94526463144462,"y":139.60901875353233,"color":"a"},{"x":196.28316775972647,"y":142.05141090824873,"color":"a"},{"x":194.89916980040442,"y":116.31456359986026,"color":"a"},{"x":210.07723121250433,"y":93.04224320152605,"color":"a"},{"x":243.82873301572303,"y":147.6762295028479,"color":"a"},{"x":263.98843551104545,"y":149.852259741618,"color":"a"},{"x":259.4691755514938,"y":144.43875437842757,"color":"a"},{"x":267.7601780459284,"y":122.70403583085567,"color":"a"},{"x":310.07318417338297,"y":153.5299505008761,"color":"a"},{"x":255.93031104904267,"y":150.31683548365686,"color":"a"},{"x":334.33142489417446,"y":177.98401524565787,"color":"a"},{"x":310.2094715058291,"y":178.81353999872533,"color":"a"},{"x":306.12091238944413,"y":185.11052652000058,"color":"a"},{"x":324.9767113442275,"y":197.8677757497257,"color":"a"},{"x":322.2458114333297,"y":185.11460731536152,"color":"a"},{"x":305.08689667571997,"y":190.11019951176655,"color":"a"},{"x":330.7128986394869,"y":218.35130844193264,"color":"a"},{"x":301.8451265147037,"y":222.29013863164244,"color":"a"},{"x":339.3318668827468,"y":222.32902170134804,"color":"a"},{"x":350.495429886857,"y":205.06011172619884,"color":"a"},{"x":296.4417608821833,"y":250.18423173783026,"color":"a"},{"x":302.1917194890704,"y":264.2242498235049,"color":"a"},{"x":309.8772510226656,"y":295.79463854169694,"color":"a"},{"x":349.624725651066,"y":338.35789943157477,"color":"a"},{"x":303.8046617172041,"y":288.9169005530768,"color":"a"},{"x":347.0535797264131,"y":336.87494253080916,"color":"a"},{"x":275.5268236592915,"y":338.0865705064459,"color":"a"},{"x":353.15495068518874,"y":333.39698042427307,"color":"a"},{"x":283.91914724937567,"y":346.5680545115347,"color":"a"},{"x":333.9099215690961,"y":360.62090772612936,"color":"a"},{"x":376.882448858889,"y":390.661333689958,"color":"a"},{"x":323.13623088017533,"y":380.5495334615267,"color":"a"},{"x":319.23121576328,"y":376.87636509540005,"color":"a"},{"x":321.24360622184423,"y":381.31772366300464,"color":"a"},{"x":315.01747506323306,"y":384.8570786619182,"color":"a"},{"x":320.03679121558304,"y":323.40829666717696,"color":"a"},{"x":320.3383189856284,"y":330.98210962127564,"color":"a"},{"x":304.58385369763164,"y":373.0590049753147,"color":"a"},{"x":363.7428317674243,"y":403.2344462176031,"color":"a"},{"x":280.71142279047837,"y":373.45878196864373,"color":"a"},{"x":291.3899623659736,"y":358.9217716399396,"color":"a"},{"x":325.0339239481714,"y":404.1792747105984,"color":"a"},{"x":293.9609456946626,"y":404.88880369183806,"color":"a"},{"x":313.9345276452363,"y":390.21025884638607,"color":"a"},{"x":271.30929944009284,"y":411.37024604148843,"color":"a"},{"x":261.1130346197553,"y":414.82060265627234,"color":"a"},{"x":274.08984152647685,"y":362.69346439994024,"color":"a"},{"x":287.19340819096476,"y":411.37590912402527,"color":"a"},{"x":275.7876433789665,"y":387.31256948725667,"color":"a"},{"x":228.6237642245018,"y":389.45744111282977,"color":"a"},{"x":252.99375204266227,"y":406.1772501296368,"color":"a"},{"x":222.45202881678267,"y":393.4641826879756,"color":"a"},{"x":255.795509591857,"y":356.3235234921025,"color":"a"},{"x":275.7065702319685,"y":351.1503598400096,"color":"a"},{"x":232.13848928317145,"y":396.67582598227466,"color":"a"},{"x":216.5413232375233,"y":402.9506318525161,"color":"a"},{"x":210.76082337498983,"y":399.7633311754264,"color":"a"},{"x":194.53955447461618,"y":394.15775280437975,"color":"a"},{"x":148.403952233451,"y":393.80839691673486,"color":"a"},{"x":143.77408875470414,"y":412.3700360543163,"color":"a"},{"x":159.27395672571362,"y":399.5222742006099,"color":"a"},{"x":119.23070460061014,"y":421.0368858932586,"color":"a"},{"x":132.41828894869226,"y":440.2700859739129,"color":"a"},{"x":151.09938566038323,"y":398.2470686787153,"color":"a"},{"x":68.75498974269362,"y":383.6057097344095,"color":"a"},{"x":112.64199455016231,"y":406.2314975734317,"color":"a"},{"x":116.79565413091163,"y":423.39830476013935,"color":"a"},{"x":105.47703892695561,"y":422.97746850923147,"color":"a"},{"x":156.85844310681802,"y":384.89765320960043,"color":"a"},{"x":39.59826688881154,"y":337.39959963118224,"color":"a"},{"x":140.89146020483375,"y":357.44233195536174,"color":"a"},{"x":60.3661249194454,"y":405.1413848545351,"color":"a"},{"x":60.31373312045298,"y":400.5356839540838,"color":"a"},{"x":62.95809512286538,"y":357.5392526098697,"color":"a"},{"x":98.96598639632899,"y":301.79112943864914,"color":"a"},{"x":100.49155875446169,"y":383.82540334517046,"color":"a"},{"x":109.14249258197893,"y":333.735570225115,"color":"a"},{"x":29.831154648014625,"y":332.32752344213463,"color":"a"},{"x":67.81001376850593,"y":354.05963771254255,"color":"a"},{"x":55.621748069385674,"y":310.5434599172713,"color":"a"},{"x":51.57249080307831,"y":288.53165984709415,"color":"a"},{"x":31.487102233979385,"y":237.9499846749219,"color":"a"},{"x":66.87510499907138,"y":320.2863206265344,"color":"a"},{"x":93.9033540546286,"y":259.10626257029,"color":"a"},{"x":92.02386288469384,"y":291.664000420661,"color":"a"},{"x":81.1039117976834,"y":238.60771321435368,"color":"a"},{"x":57.040354371791366,"y":230.74663133122954,"color":"a"},{"x":91.64944768118076,"y":244.88339658307763,"color":"a"},{"x":92.33770341050071,"y":238.9600310426227,"color":"a"},{"x":128.72331702100792,"y":244.8883023512787,"color":"a"},{"x":38.941655306305336,"y":283.74003214447475,"color":"a"},{"x":58.665064643791794,"y":231.28649504618437,"color":"a"},{"x":76.26904834289711,"y":236.03057444035693,"color":"a"},{"x":61.04839236259383,"y":243.13664402401105,"color":"a"},{"x":106.35946051559799,"y":203.09854163114687,"color":"a"},{"x":84.97684094979319,"y":206.49999467240212,"color":"a"},{"x":110.28686468525362,"y":252.23857871301217,"color":"a"},{"x":49.12813926827531,"y":243.68376704394694,"color":"a"},{"x":61.512741119645085,"y":209.50290128242875,"color":"a"},{"x":83.1987468695908,"y":221.8874156142631,"color":"a"},{"x":54.334153800384726,"y":225.44041312599785,"color":"a"},{"x":86.08596567866424,"y":211.85931658923693,"color":"a"},{"x":96.73175547449267,"y":237.8166554679135,"color":"a"},{"x":133.52711313585752,"y":244.85401179017003,"color":"a"},{"x":82.55711989247413,"y":233.38172129825682,"color":"a"},{"x":109.11615560850406,"y":178.1719852641565,"color":"a"},{"x":50.92260531357596,"y":191.09564578884562,"color":"a"},{"x":119.83754309982079,"y":185.32889695801936,"color":"a"},{"x":103.2996947687753,"y":224.19619093467833,"color":"a"},{"x":24.99845162322633,"y":178.7027798291138,"color":"a"},{"x":165.68330394017812,"y":164.1251898431329,"color":"a"},{"x":97.07846802710455,"y":174.07649139665472,"color":"a"},{"x":90.9604070502217,"y":187.77068357753512,"color":"a"},{"x":139.6517102140642,"y":181.99726613069447,"color":"a"},{"x":103.28319267786037,"y":164.2438133583268,"color":"a"},{"x":131.91625578120747,"y":125.90645308972933,"color":"a"},{"x":135.6625826997096,"y":169.75480486985612,"color":"a"},{"x":168.22658971775382,"y":146.1244562572839,"color":"a"},{"x":142.16128410725167,"y":150.45242790266644,"color":"a"},{"x":145.52207184014236,"y":121.8203621885313,"color":"a"},{"x":145.94214653835073,"y":131.308483979887,"color":"a"},{"x":164.23511631655285,"y":108.01980309249564,"color":"a"},{"x":162.70110470239587,"y":155.06642869655843,"color":"a"},{"x":108.37771905514697,"y":191.58712505028228,"color":"a"},{"x":176.62214937605367,"y":113.02961842596113,"color":"a"},{"x":184.16908839512544,"y":132.20931856755448,"color":"a"},{"x":183.44968761525098,"y":134.1213551255292,"color":"a"},{"x":202.4518564598128,"y":158.3287134999248,"color":"a"},{"x":197.7009059510665,"y":168.4128753767106,"color":"a"},{"x":221.4265811753853,"y":144.19800454939997,"color":"a"},{"x":224.0839410076748,"y":114.54013287383395,"color":"a"},{"x":188.44547492056137,"y":126.4173294478868,"color":"a"},{"x":194.32453087838778,"y":171.6998094779185,"color":"a"},{"x":220.0921678404616,"y":97.77046342379316,"color":"a"},{"x":246.33839174191843,"y":150.4625500975767,"color":"a"},{"x":241.3969203760561,"y":96.87420288646194,"color":"a"},{"x":270.8751873134608,"y":106.82868109448293,"color":"a"},{"x":297.94279946696065,"y":144.85397798811914,"color":"a"},{"x":202.0541278786656,"y":101.58572682814139,"color":"a"},{"x":255.38833579575092,"y":114.807315306374,"color":"a"},{"x":252.27335593451346,"y":110.9539368890488,"color":"a"},{"x":281.2104258589908,"y":126.94126810019537,"color":"a"},{"x":263.63340814062184,"y":119.80271333404755,"color":"a"},{"x":259.47502689212286,"y":147.46104188885027,"color":"a"},{"x":269.6809089341253,"y":105.2547332592718,"color":"a"},{"x":232.75166218418906,"y":174.99028650883355,"color":"a"},{"x":227.76084085837593,"y":166.06516645099373,"color":"a"},{"x":261.9156533178665,"y":137.0375736362168,"color":"a"},{"x":274.91270216914484,"y":146.86933489892562,"color":"a"},{"x":279.080180298723,"y":139.54491284939968,"color":"a"},{"x":305.7373383205417,"y":141.2915140913874,"color":"a"},{"x":282.5342558138515,"y":170.12828490959362,"color":"a"},{"x":287.45709254244446,"y":150.40249906408366,"color":"a"},{"x":321.0057509549267,"y":244.8566310385492,"color":"a"},{"x":310.47116462447417,"y":141.47872687470073,"color":"a"},{"x":287.78013506386606,"y":170.40314673827777,"color":"a"},{"x":287.24246695301144,"y":176.5395617348358,"color":"a"},{"x":285.9745471366226,"y":166.0885064099332,"color":"a"},{"x":320.3331583188587,"y":227.0872887087985,"color":"a"},{"x":298.7533020653839,"y":170.25907913462243,"color":"a"},{"x":326.3573954597764,"y":185.59870442241737,"color":"a"},{"x":355.61597935339285,"y":255.96843276918966,"color":"a"},{"x":332.28255117526606,"y":179.96867214841194,"color":"a"},{"x":340.780655682105,"y":238.65289489679827,"color":"a"},{"x":361.41733842879984,"y":246.9291499935373,"color":"a"},{"x":318.39322189271786,"y":265.9959008657918,"color":"a"},{"x":359.2228507117886,"y":240.74878582724335,"color":"a"},{"x":346.56089857914094,"y":218.83297928496563,"color":"a"},{"x":291.836455324819,"y":250.98717168580237,"color":"a"},{"x":295.546366466021,"y":333.5997542288185,"color":"a"},{"x":333.15146129837194,"y":287.2661110834623,"color":"a"},{"x":322.29964963522303,"y":263.374339572703,"color":"a"},{"x":340.4050296270798,"y":252.5416167931631,"color":"a"},{"x":338.80779163046896,"y":284.271543828736,"color":"a"},{"x":353.1850539714747,"y":315.56624759648184,"color":"a"},{"x":304.2306772851674,"y":325.2861829881283,"color":"a"},{"x":289.71488911291397,"y":274.47876514290164,"color":"a"},{"x":340.9848957250739,"y":308.3931839456552,"color":"a"},{"x":332.29115085624676,"y":316.3294526136298,"color":"a"},{"x":362.5425691405135,"y":334.97340194216895,"color":"a"},{"x":335.61569274300336,"y":358.32696059015854,"color":"a"},{"x":311.68735619836383,"y":374.4926580829378,"color":"a"},{"x":342.860801442799,"y":332.24635944786735,"color":"a"},{"x":343.89734958732834,"y":337.5825064051738,"color":"a"},{"x":330.63547579120655,"y":327.26626759446384,"color":"a"},{"x":291.7616224624376,"y":377.71675793441375,"color":"a"},{"x":327.8279034382453,"y":378.3354567369079,"color":"a"},{"x":316.07029729976523,"y":362.32272864361255,"color":"a"},{"x":331.4320529639639,"y":378.42055572939597,"color":"a"},{"x":366.30171498843686,"y":332.84361663684706,"color":"a"},{"x":331.4240807613642,"y":386.83827467772835,"color":"a"},{"x":320.96994958426046,"y":372.2453464864818,"color":"a"},{"x":353.5171438965675,"y":383.78282880121316,"color":"a"},{"x":350.5779018941448,"y":356.37564549117417,"color":"a"},{"x":342.9673980682254,"y":403.61466109696363,"color":"a"},{"x":312.99080866340927,"y":361.66130006432275,"color":"a"},{"x":257.83654613703754,"y":447.0504873699381,"color":"a"},{"x":232.44798997873997,"y":261.92709341428144,"color":"b"},{"x":203.64138294589281,"y":213.9059726520448,"color":"b"},{"x":175.54417136984569,"y":263.69270776185147,"color":"b"},{"x":204.46149757034706,"y":295.27735498250024,"color":"b"},{"x":219.5849819029156,"y":266.7634759328999,"color":"b"},{"x":258.86919275936344,"y":241.23165121926667,"color":"b"},{"x":201.8265877206688,"y":278.04829858885284,"color":"b"},{"x":190.1134496509043,"y":278.70926337399146,"color":"b"},{"x":213.96924226310603,"y":250.59491358578987,"color":"b"},{"x":235.4557859277626,"y":304.58095206613245,"color":"b"},{"x":206.19996384897289,"y":311.8655107077967,"color":"b"},{"x":162.2336578042759,"y":318.6773058733602,"color":"b"},{"x":238.16758528700387,"y":281.4172104695009,"color":"b"},{"x":177.1329173861631,"y":313.3325522138873,"color":"b"},{"x":197.54495854222031,"y":266.40767363113576,"color":"b"},{"x":218.3967211951555,"y":262.19579468963735,"color":"b"},{"x":172.70593003324765,"y":267.6387829900528,"color":"b"},{"x":219.58668135705554,"y":285.2964491654292,"color":"b"},{"x":227.7846765007434,"y":316.9091483534503,"color":"b"},{"x":264.94043163535457,"y":321.13893316731856,"color":"b"},{"x":194.30194164791206,"y":230.54312692569408,"color":"b"},{"x":253.39746227268552,"y":270.1116079464824,"color":"b"},{"x":229.9600899345265,"y":284.4446651852818,"color":"b"},{"x":215.41904870978385,"y":312.419793725546,"color":"b"},{"x":274.50731969967563,"y":243.27458376815338,"color":"b"},{"x":242.46483293985517,"y":268.37537444470377,"color":"b"},{"x":191.6277495542905,"y":279.3720369895712,"color":"b"},{"x":224.7406630230693,"y":246.73846210800482,"color":"b"},{"x":199.66842494587442,"y":235.0580923704489,"color":"b"},{"x":191.49605420475498,"y":244.97095612161107,"color":"b"},{"x":195.43863771796504,"y":284.2443361916327,"color":"b"},{"x":212.76102248496883,"y":281.43030073859086,"color":"b"},{"x":218.7102796126658,"y":261.24852891131115,"color":"b"},{"x":185.49824699054855,"y":266.35378682158444,"color":"b"},{"x":238.11904210767187,"y":259.05958755170155,"color":"b"},{"x":207.44346516203854,"y":249.76795281988026,"color":"b"},{"x":195.33148270621433,"y":252.07042655106022,"color":"b"},{"x":198.3329462539883,"y":288.6473041870154,"color":"b"},{"x":254.39063670388845,"y":300.1828636954875,"color":"b"},{"x":156.95078143577513,"y":282.36597482697465,"color":"b"},{"x":154.76500806294143,"y":272.13850152196574,"color":"b"},{"x":227.31757103440393,"y":264.63591475217936,"color":"b"},{"x":209.488862727582,"y":274.25333693766163,"color":"b"},{"x":220.18950433641044,"y":271.9945109909827,"color":"b"},{"x":184.21462345487143,"y":321.23775285409954,"color":"b"},{"x":182.0982584523742,"y":267.79537681964786,"color":"b"},{"x":171.53102822722477,"y":267.73088066819787,"color":"b"},{"x":180.6488849093438,"y":254.08382921134574,"color":"b"},{"x":233.9434241501984,"y":286.79772826135354,"color":"b"},{"x":161.17606928637534,"y":299.40296632629145,"color":"b"},{"x":238.1763507512551,"y":292.2732662961113,"color":"b"},{"x":237.8288303974304,"y":284.4871117565276,"color":"b"},{"x":202.48415446981474,"y":287.34569653186225,"color":"b"},{"x":204.85595164743424,"y":245.99893710435018,"color":"b"},{"x":201.82013927126243,"y":255.1185126913113,"color":"b"},{"x":248.572899871328,"y":297.48941507167837,"color":"b"},{"x":237.53397364702028,"y":232.33057156426747,"color":"b"},{"x":171.26159760138268,"y":256.2912893865456,"color":"b"},{"x":213.24463572120175,"y":245.62173330308738,"color":"b"},{"x":177.10864285346827,"y":294.17451187287236,"color":"b"},{"x":228.18717145905038,"y":212.55862004260558,"color":"b"},{"x":194.29711952915787,"y":273.75187054035047,"color":"b"},{"x":200.2850273388829,"y":246.9560855927677,"color":"b"},{"x":212.14652890681927,"y":280.8261541399296,"color":"b"},{"x":228.01427918462656,"y":301.38636800207854,"color":"b"},{"x":204.8221366972792,"y":231.73543775482426,"color":"b"},{"x":268.76235978991633,"y":284.99168423294657,"color":"b"},{"x":243.35326399043163,"y":282.6681834065896,"color":"b"},{"x":164.8131311218159,"y":249.57871412445914,"color":"b"},{"x":178.13142564001478,"y":254.37548723056085,"color":"b"},{"x":256.4367734557962,"y":257.53349498758405,"color":"b"},{"x":227.93968523253463,"y":276.5253190335297,"color":"b"},{"x":223.99802304641128,"y":254.13930985399736,"color":"b"},{"x":191.44055014714363,"y":306.33757858848054,"color":"b"},{"x":223.3474932206692,"y":264.3667466890168,"color":"b"},{"x":198.0673474368756,"y":285.12933735538974,"color":"b"},{"x":216.55874847826564,"y":282.9650833588588,"color":"b"},{"x":251.8858033329343,"y":240.2700448949338,"color":"b"},{"x":188.17086826983987,"y":259.81475422033384,"color":"b"},{"x":213.48588687009243,"y":265.6724012842935,"color":"b"},{"x":200.43761635899327,"y":275.4313059421247,"color":"b"},{"x":167.0974939836974,"y":314.5825601391324,"color":"b"},{"x":212.78230816149184,"y":267.8283910258627,"color":"b"},{"x":192.89775198493766,"y":258.49459662954973,"color":"b"},{"x":214.63850911078617,"y":277.8813668661494,"color":"b"},{"x":240.40489615751315,"y":270.1091843781355,"color":"b"},{"x":183.35733368902007,"y":266.2181811329558,"color":"b"},{"x":188.42962080588478,"y":245.22129846904767,"color":"b"},{"x":135.59602767808772,"y":264.6072439540261,"color":"b"},{"x":232.78091923702306,"y":284.21463554632703,"color":"b"},{"x":258.3639780318904,"y":233.4926378987414,"color":"b"},{"x":182.71769543955105,"y":286.4394450717878,"color":"b"},{"x":202.80289405229195,"y":264.93896332256804,"color":"b"},{"x":186.12545499793706,"y":285.4389165234638,"color":"b"},{"x":240.55383490547115,"y":287.98165336860535,"color":"b"},{"x":240.13492864003095,"y":219.42945705345926,"color":"b"},{"x":232.04874591105957,"y":283.09202101209974,"color":"b"},{"x":198.1257089567862,"y":278.5271136975023,"color":"b"},{"x":210.7648010569444,"y":289.57957613019875,"color":"b"},{"x":226.27248087762922,"y":233.4984173490853,"color":"b"},{"x":213.4621752137002,"y":257.5121814370688,"color":"b"},{"x":194.2909241726106,"y":293.0373226069729,"color":"b"},{"x":240.86921698939256,"y":261.74719362686454,"color":"b"},{"x":217.24399047521555,"y":270.293788372847,"color":"b"},{"x":214.8637598308626,"y":255.0773033034042,"color":"b"},{"x":226.98364684752562,"y":286.9389180792815,"color":"b"},{"x":211.9252458769277,"y":309.565408537842,"color":"b"},{"x":216.48643252365372,"y":232.52792817176464,"color":"b"},{"x":189.57854807276863,"y":303.54128559316143,"color":"b"},{"x":168.4411425472501,"y":288.7598287416988,"color":"b"},{"x":225.0382798086131,"y":229.44467060677096,"color":"b"},{"x":227.30428099096082,"y":274.0390333457385,"color":"b"},{"x":172.8643078118302,"y":248.57140818759544,"color":"b"},{"x":206.96281392537693,"y":298.9578566292439,"color":"b"},{"x":152.10886407292153,"y":247.58387520039125,"color":"b"},{"x":231.3759618832396,"y":271.179606851087,"color":"b"},{"x":239.90360971093523,"y":294.20177537983056,"color":"b"},{"x":214.59350759293127,"y":286.46567879368945,"color":"b"},{"x":204.4719675794109,"y":259.6433508002754,"color":"b"},{"x":165.41262950277758,"y":274.8829183091907,"color":"b"},{"x":178.72885512280902,"y":299.5034818918718,"color":"b"},{"x":212.4273427327209,"y":266.0713981972312,"color":"b"},{"x":180.01561331373568,"y":273.57770238305386,"color":"b"},{"x":235.7042772477202,"y":314.2049955268874,"color":"b"},{"x":206.8156981091991,"y":237.42155886567394,"color":"b"},{"x":173.88396880405855,"y":349.1909965108747,"color":"b"},{"x":234.46244150285864,"y":391.8157950860415,"color":"a"},{"x":258.0469275255645,"y":382.73552412269044,"color":"a"},{"x":248.60229605217484,"y":425.95482677913947,"color":"a"},{"x":225.77864530758657,"y":381.5902027235519,"color":"a"},{"x":234.4493190036398,"y":391.44184996478384,"color":"a"},{"x":248.96305878596502,"y":380.9001109122073,"color":"a"},{"x":249.62962935983884,"y":338.01356020479875,"color":"a"},{"x":230.17495568563368,"y":394.21240341155874,"color":"a"},{"x":214.57243929038765,"y":387.14909137160475,"color":"a"},{"x":224.6974077737128,"y":367.2943949854414,"color":"a"},{"x":247.96988263848456,"y":408.05199319483535,"color":"a"},{"x":208.11649228703004,"y":381.0439577598422,"color":"a"},{"x":227.07536128070382,"y":370.34228593818113,"color":"a"},{"x":214.3550601439432,"y":435.27820250433786,"color":"a"},{"x":179.19307572804206,"y":385.9769733049941,"color":"a"},{"x":170.20239047415285,"y":377.6451647354378,"color":"a"},{"x":191.38943100987515,"y":359.56372367398296,"color":"a"},{"x":186.9361594064543,"y":390.1896709483681,"color":"a"},{"x":136.52322583286463,"y":367.0649625677234,"color":"a"},{"x":151.19419584317632,"y":379.65148754286287,"color":"a"},{"x":167.73404656555766,"y":444.2293565743783,"color":"a"},{"x":121.66738974089834,"y":370.7983881272739,"color":"a"},{"x":143.1868413631922,"y":384.53012089731556,"color":"a"},{"x":147.48566991102075,"y":399.7975003270932,"color":"a"},{"x":128.1505178198951,"y":408.784554358549,"color":"a"},{"x":133.50369750835796,"y":388.96657056120085,"color":"a"},{"x":151.02726516098096,"y":351.6247177190159,"color":"a"},{"x":124.93962724267244,"y":353.50244361936313,"color":"a"},{"x":124.79933362435249,"y":391.57789778775486,"color":"a"},{"x":156.9950892837756,"y":398.09701415686584,"color":"a"},{"x":121.15716355460475,"y":353.96036508329985,"color":"a"},{"x":124.42848948739349,"y":403.9980338114375,"color":"a"},{"x":130.5494799733484,"y":341.3265709929215,"color":"a"},{"x":124.887285785201,"y":323.3053218782603,"color":"a"},{"x":56.49603664119188,"y":331.1433130579453,"color":"a"},{"x":78.85537152929881,"y":313.78615505551346,"color":"a"},{"x":85.35468914653072,"y":320.86401683108477,"color":"a"},{"x":121.96719285738828,"y":404.20538707579806,"color":"a"},{"x":106.76314318902101,"y":349.7128597790077,"color":"a"},{"x":45.47794920705,"y":417.1019947232409,"color":"a"},{"x":82.54875077178842,"y":346.7902168154451,"color":"a"},{"x":84.96652259525116,"y":351.38757056066873,"color":"a"},{"x":66.50923838452267,"y":374.14308019731345,"color":"a"},{"x":66.41281567800871,"y":338.28226610934496,"color":"a"},{"x":103.30114089670738,"y":323.05128012881516,"color":"a"},{"x":87.45473350776767,"y":369.0933728816986,"color":"a"},{"x":105.70679658779875,"y":350.06995255734427,"color":"a"},{"x":75.82875026298144,"y":302.3714081158048,"color":"a"},{"x":85.56047507500699,"y":317.6431668943155,"color":"a"},{"x":11.1643352821239,"y":337.13574487348717,"color":"a"},{"x":65.61904866921769,"y":331.67469448412993,"color":"a"},{"x":93.6711885414449,"y":289.51570289277055,"color":"a"},{"x":111.84586066037068,"y":316.98141098858605,"color":"a"},{"x":39.637995653142,"y":295.15285601632627,"color":"a"},{"x":96.47488762708453,"y":259.2211957866989,"color":"a"},{"x":50.60329375818419,"y":292.1668311135432,"color":"a"},{"x":96.14522690979238,"y":289.8056356977945,"color":"a"},{"x":60.884366511118046,"y":285.15859635630306,"color":"a"},{"x":76.6202596890451,"y":296.5811143070034,"color":"a"},{"x":93.82854571805117,"y":274.9143163586393,"color":"a"},{"x":84.2964454371068,"y":273.4642365170098,"color":"a"},{"x":116.63757610131236,"y":276.40129994158144,"color":"a"},{"x":81.50508129160927,"y":262.82811571719094,"color":"a"},{"x":56.97930703977871,"y":277.4879757249506,"color":"a"},{"x":104.2903365441871,"y":262.3481146391119,"color":"a"},{"x":83.42679198275853,"y":199.87929821750402,"color":"a"},{"x":68.08501640240402,"y":236.67674196434,"color":"a"},{"x":11.405750706918575,"y":290.76379168162805,"color":"a"},{"x":52.60662749848605,"y":266.3371390391111,"color":"a"},{"x":58.64741097741955,"y":260.55361583334377,"color":"a"},{"x":97.11396768815764,"y":275.87206339542547,"color":"a"},{"x":-0.055875713308765285,"y":221.59987939055583,"color":"a"},{"x":62.47728208425233,"y":267.334209080868,"color":"a"},{"x":76.97246095304597,"y":189.8382609193422,"color":"a"},{"x":71.71337056213186,"y":221.97191336399106,"color":"a"},{"x":81.5030225236576,"y":219.72450534696924,"color":"a"},{"x":70.0318854246353,"y":219.82559747084036,"color":"a"},{"x":42.39465604027238,"y":171.27684259326247,"color":"a"},{"x":36.47433057013388,"y":233.15877262185575,"color":"a"},{"x":106.42351434392944,"y":222.25248521276131,"color":"a"},{"x":82.48743248158235,"y":191.93234887346932,"color":"a"},{"x":29.075665705796453,"y":216.48400121403074,"color":"a"},{"x":75.29660920872266,"y":194.70009109057503,"color":"a"},{"x":105.24120493285827,"y":205.20607897865722,"color":"a"},{"x":16.52967168452163,"y":179.58102343266074,"color":"a"},{"x":53.151686824501596,"y":167.15329193494796,"color":"a"},{"x":97.84817309778398,"y":159.5820005578367,"color":"a"},{"x":91.85319820527744,"y":170.35231182087398,"color":"a"},{"x":95.29327839510258,"y":182.97220360363423,"color":"a"},{"x":40.98716081111995,"y":153.06913233654547,"color":"a"},{"x":48.95776991411568,"y":169.8196134441023,"color":"a"},{"x":81.55167158283115,"y":158.3037141901853,"color":"a"},{"x":76.92364328940417,"y":102.1774440044249,"color":"a"},{"x":77.33443848418322,"y":149.88940766712318,"color":"a"},{"x":120.45663527025233,"y":178.4704378553165,"color":"a"},{"x":92.95747287912701,"y":152.04784805379285,"color":"a"},{"x":87.58294344627701,"y":159.35557298354365,"color":"a"},{"x":120.62186180797028,"y":142.4925427317154,"color":"a"},{"x":116.30684340038405,"y":151.93326246450727,"color":"a"},{"x":112.89431993278589,"y":126.60628300975992,"color":"a"},{"x":91.21110499610086,"y":146.3296473612172,"color":"a"},{"x":76.51798565740327,"y":145.1292542931712,"color":"a"},{"x":151.28425415174564,"y":122.16808274450278,"color":"a"},{"x":142.8770028761796,"y":172.84437941571468,"color":"a"},{"x":140.1049063305943,"y":136.1033701806732,"color":"a"},{"x":186.81483743542682,"y":155.64470330068536,"color":"a"},{"x":144.33137181313347,"y":122.51511856574251,"color":"a"},{"x":155.49896282549818,"y":132.70368029747544,"color":"a"},{"x":149.8973256415534,"y":162.37518768394477,"color":"a"},{"x":127.01351757238535,"y":120.95067257471317,"color":"a"},{"x":173.25223929093661,"y":104.85878697799319,"color":"a"},{"x":168.9683424595134,"y":96.12470687139296,"color":"a"},{"x":204.1542932679157,"y":118.22298245572875,"color":"a"},{"x":144.8262855520628,"y":121.64400086503656,"color":"a"},{"x":192.29543666413537,"y":132.53527401135977,"color":"a"},{"x":187.34141914350042,"y":145.2652548931154,"color":"a"},{"x":257.0913733385002,"y":150.51792851819317,"color":"a"},{"x":156.57017304269692,"y":84.60229628505869,"color":"a"},{"x":243.0397057879448,"y":109.09924877506614,"color":"a"},{"x":250.6522425990782,"y":114.1944019395417,"color":"a"},{"x":229.64397404934093,"y":121.69630379848388,"color":"a"},{"x":247.0794664422754,"y":105.47690689105855,"color":"a"},{"x":284.9919464496495,"y":164.68084965776683,"color":"a"},{"x":221.36489875509852,"y":124.46130515737565,"color":"a"},{"x":221.42252501936443,"y":152.99451324917686,"color":"a"},{"x":249.50990569246335,"y":170.00226428886128,"color":"a"},{"x":239.49452557848522,"y":131.59462290494872,"color":"a"},{"x":300.4233062622853,"y":143.9215671391927,"color":"a"},{"x":212.69735221279888,"y":87.61057201229278,"color":"a"},{"x":222.39924038864064,"y":136.38718399307294,"color":"a"},{"x":280.9125812262152,"y":141.77554549070243,"color":"a"},{"x":255.89431988437525,"y":91.58022024118753,"color":"a"},{"x":281.5737474102064,"y":108.15846504399269,"color":"a"},{"x":256.1032236315197,"y":133.06901490272116,"color":"a"},{"x":280.4339791599244,"y":166.28425504129478,"color":"a"},{"x":329.585075745071,"y":146.0936509856844,"color":"a"},{"x":288.97767095568986,"y":181.5683194733943,"color":"a"},{"x":281.6320610904324,"y":156.86252218937392,"color":"a"},{"x":326.490516380567,"y":115.86212356771921,"color":"a"},{"x":305.53385920405236,"y":131.22251408462068,"color":"a"},{"x":322.83704521396874,"y":163.8902959801315,"color":"a"},{"x":348.05456622917393,"y":133.1111780770189,"color":"a"},{"x":310.20996947808044,"y":154.6904871467794,"color":"a"},{"x":336.00624291839796,"y":146.89011320663366,"color":"a"},{"x":333.3062576410729,"y":182.17286460954227,"color":"a"},{"x":304.8904230838901,"y":173.3432608503013,"color":"a"},{"x":281.16613943131136,"y":164.59901803217917,"color":"a"},{"x":369.1347391876613,"y":145.70656472192462,"color":"a"},{"x":344.7638838602301,"y":197.59467620062105,"color":"a"},{"x":329.3165397370617,"y":173.51971430803735,"color":"a"},{"x":283.8026507238578,"y":165.73272157657158,"color":"a"},{"x":372.9113965137628,"y":165.3738751736447,"color":"a"},{"x":341.5819192502487,"y":159.5935419955934,"color":"a"},{"x":353.5730063995477,"y":197.4871714641468,"color":"a"},{"x":374.8076441300701,"y":219.64893719505375,"color":"a"},{"x":341.34460703572785,"y":238.73013584082867,"color":"a"},{"x":336.70542019395737,"y":231.83583838305054,"color":"a"},{"x":342.8613034991948,"y":214.26832736581338,"color":"a"},{"x":359.21931957866695,"y":223.09721059045165,"color":"a"},{"x":345.2520772218167,"y":203.38798878675607,"color":"a"},{"x":316.72727591074516,"y":253.25941220665186,"color":"a"},{"x":386.27926685856875,"y":217.7548909172292,"color":"a"},{"x":369.65964003971305,"y":180.2817099980477,"color":"a"},{"x":390.3186962969136,"y":291.2665465150211,"color":"a"},{"x":340.51614347955746,"y":271.43204676782534,"color":"a"},{"x":327.8062368204933,"y":288.62307722471036,"color":"a"},{"x":343.249354406928,"y":283.12370541331535,"color":"a"},{"x":377.31593733708416,"y":281.87501447991923,"color":"a"},{"x":343.6470356410392,"y":343.2430070989789,"color":"a"},{"x":358.7603464857126,"y":316.77769693799985,"color":"a"},{"x":356.2888326761513,"y":305.4155586030778,"color":"a"},{"x":303.17511440749144,"y":272.36349315065036,"color":"a"},{"x":347.9154519480569,"y":301.494358817919,"color":"a"},{"x":343.8124082536146,"y":316.75001435134175,"color":"a"},{"x":302.6480471511157,"y":340.26629383167267,"color":"a"},{"x":357.055274402917,"y":326.3618274483696,"color":"a"},{"x":371.67068245824055,"y":304.37595829721573,"color":"a"},{"x":358.3271607114771,"y":317.21342357456365,"color":"a"},{"x":361.89308733383047,"y":330.23361384439573,"color":"a"},{"x":322.44252218268196,"y":333.52951327819335,"color":"a"},{"x":360.1113066546599,"y":415.18564850933103,"color":"a"},{"x":357.43172087909454,"y":321.1713394836477,"color":"a"},{"x":373.59600924578564,"y":328.6080379206407,"color":"a"},{"x":345.7142845648166,"y":346.16260339346684,"color":"a"},{"x":372.91456175625166,"y":382.1507248329592,"color":"a"},{"x":360.3447747664423,"y":375.0041562980988,"color":"a"},{"x":325.58279638507634,"y":343.93776114389277,"color":"a"},{"x":322.19146388622136,"y":384.4127023887746,"color":"a"},{"x":314.0368679538866,"y":360.6191547095429,"color":"a"},{"x":338.71911099672354,"y":364.73929093592767,"color":"a"},{"x":322.3172990486928,"y":343.308275692502,"color":"a"},{"x":349.77260551915253,"y":348.95305830023483,"color":"a"},{"x":301.09516079836857,"y":359.5952507328666,"color":"a"},{"x":283.97190420739,"y":399.97821809823404,"color":"a"},{"x":259.4719339392105,"y":396.40259280174826,"color":"a"},{"x":236.8806476617502,"y":383.95753087271584,"color":"a"},{"x":421.7922384531875,"y":441.42505133505017,"color":"c"},{"x":443.3368785744686,"y":472.7924903185278,"color":"c"},{"x":425.1863040506805,"y":525.1601723445616,"color":"c"},{"x":458.6042443996229,"y":449.80908355025906,"color":"c"},{"x":480.94632749104966,"y":445.8521447499161,"color":"c"},{"x":443.2614448266494,"y":446.6146230484898,"color":"c"},{"x":429.21025076294865,"y":441.5943549907558,"color":"c"},{"x":453.8520000815556,"y":431.99241576620807,"color":"c"},{"x":437.38734121162275,"y":403.240576554551,"color":"c"},{"x":393.1616323333044,"y":433.5289011916857,"color":"c"},{"x":465.95480902879206,"y":375.61177135527396,"color":"c"},{"x":438.6279933688708,"y":400.4368299573598,"color":"c"},{"x":494.94852632207534,"y":346.17787936695726,"color":"c"},{"x":505.4608843979591,"y":378.4413984468158,"color":"c"},{"x":485.30882740958975,"y":365.7233389756723,"color":"c"},{"x":470.2032104607175,"y":305.06455474556986,"color":"c"},{"x":508.66124818359896,"y":335.85654657940984,"color":"c"},{"x":479.6866426169983,"y":341.77177856035905,"color":"c"},{"x":504.19844298934197,"y":382.1629451520916,"color":"c"},{"x":516.9499718266782,"y":334.87850755015745,"color":"c"},{"x":527.3882920716986,"y":275.7515402044595,"color":"c"},{"x":525.5512135823474,"y":316.8188116507428,"color":"c"},{"x":510.23291830822217,"y":316.16143981109695,"color":"c"},{"x":531.3028227171671,"y":286.9900619508258,"color":"c"},{"x":550.3581891567405,"y":308.2402000091373,"color":"c"},{"x":530.1884996000288,"y":289.17588218438084,"color":"c"},{"x":545.6417429870066,"y":265.3679699850993,"color":"c"},{"x":577.285787184,"y":236.16933856042044,"color":"c"},{"x":576.4894586455013,"y":241.10601571944505,"color":"c"},{"x":594.3190986405807,"y":294.7960753118946,"color":"c"},{"x":570.6836273327514,"y":255.67562548844083,"color":"c"},{"x":596.3700139344995,"y":234.71025363585943,"color":"c"},{"x":580.465014034059,"y":170.5073785546137,"color":"c"},{"x":565.5693092990404,"y":201.58617147212226,"color":"c"},{"x":586.1613419461828,"y":174.96351195711384,"color":"c"},{"x":622.2561014433069,"y":169.47913225091054,"color":"c"},{"x":594.3945357277278,"y":214.85567615394626,"color":"c"},{"x":567.8614877823786,"y":172.206760361595,"color":"c"},{"x":638.7542897804151,"y":192.68426563225904,"color":"c"},{"x":641.4086443651561,"y":193.57409836189368,"color":"c"},{"x":581.787734208303,"y":123.78131193959257,"color":"c"},{"x":599.1186821974042,"y":198.7854341417284,"color":"c"},{"x":602.5091360151139,"y":176.88001377384285,"color":"c"},{"x":645.198209520727,"y":169.88330494242126,"color":"c"},{"x":562.0263864075445,"y":134.2679308424095,"color":"c"},{"x":639.4423162145898,"y":204.3642255732218,"color":"c"},{"x":636.9520220255158,"y":124.67921477981082,"color":"c"},{"x":658.4291023977571,"y":166.08921005875692,"color":"c"},{"x":639.6547013845144,"y":182.08294273585568,"color":"c"},{"x":627.5693532259314,"y":145.07718351484738,"color":"c"},{"x":606.4052013735228,"y":124.0455466300175,"color":"c"},{"x":632.3733422784615,"y":126.15582002589116,"color":"c"},{"x":648.8693255728163,"y":134.0473340273894,"color":"c"},{"x":638.3225539973749,"y":146.1993646187741,"color":"c"},{"x":662.5724205185644,"y":70.30571013767837,"color":"c"},{"x":666.2223570611601,"y":97.68726040923366,"color":"c"},{"x":596.6612907842186,"y":111.50257442469172,"color":"c"},{"x":634.8265506813619,"y":117.30440463135699,"color":"c"},{"x":664.5933517697458,"y":76.43658725164875,"color":"c"},{"x":656.2783799495842,"y":67.70334769835262,"color":"c"},{"x":639.6748692909422,"y":112.46323461299227,"color":"c"},{"x":713.0578332267673,"y":84.88963201271747,"color":"c"},{"x":704.4684977709367,"y":129.2997193415673,"color":"c"},{"x":650.2456687237212,"y":66.97119283485523,"color":"c"},{"x":612.4355200061993,"y":90.27913606166976,"color":"c"},{"x":719.2880714932811,"y":72.21904688920574,"color":"c"},{"x":661.6593759606228,"y":127.95311720459028,"color":"c"},{"x":641.1278774226845,"y":55.08435199932205,"color":"c"},{"x":632.271442406451,"y":90.05082781689941,"color":"c"},{"x":705.8336105239287,"y":118.36032230465491,"color":"c"},{"x":673.3798036295314,"y":143.4801765566266,"color":"c"},{"x":665.6666643235429,"y":108.11521426959462,"color":"c"},{"x":684.7730789829282,"y":167.24693577741402,"color":"c"},{"x":620.2985488615658,"y":129.51923646927048,"color":"c"},{"x":698.8455204241221,"y":129.57977950197204,"color":"c"},{"x":651.9535995339728,"y":141.26147324221455,"color":"c"},{"x":633.3483057676248,"y":207.01067126285966,"color":"c"},{"x":640.3402453695662,"y":201.10202495537288,"color":"c"},{"x":667.3633990386251,"y":180.11504971435954,"color":"c"},{"x":568.3455896991193,"y":215.65095700474694,"color":"c"},{"x":616.4085489655262,"y":213.98503907574047,"color":"c"},{"x":618.43174886537,"y":166.04487724137147,"color":"c"},{"x":604.1813650242715,"y":228.20642205960553,"color":"c"},{"x":588.1564860527156,"y":227.71268093399897,"color":"c"},{"x":596.8219182686289,"y":213.25572853429935,"color":"c"},{"x":576.9139647121858,"y":247.597466815183,"color":"c"},{"x":572.8199506494826,"y":234.3806569599243,"color":"c"},{"x":572.3727858911648,"y":266.7502076773904,"color":"c"},{"x":587.9136487149759,"y":258.3139683423741,"color":"c"},{"x":593.0221854322845,"y":293.80043418072984,"color":"c"},{"x":530.6911251910308,"y":255.10127535741046,"color":"c"},{"x":545.3103252911237,"y":255.367776236919,"color":"c"},{"x":534.7642468812861,"y":321.37720050787664,"color":"c"},{"x":549.1321971669473,"y":304.79085256130327,"color":"c"},{"x":535.75260139878,"y":334.5290927675072,"color":"c"},{"x":517.3545449820305,"y":288.8077126750878,"color":"c"},{"x":505.00026366554306,"y":324.39303274885606,"color":"c"},{"x":519.8870099399871,"y":379.842703640566,"color":"c"},{"x":477.1486307614504,"y":334.9743390168629,"color":"c"},{"x":559.956912809293,"y":260.55649125059614,"color":"c"},{"x":459.90644506350947,"y":343.9296159739233,"color":"c"},{"x":499.273498164924,"y":383.5028738865212,"color":"c"},{"x":531.8262823392224,"y":343.9877793922017,"color":"c"},{"x":559.774202437193,"y":298.7579492896741,"color":"c"},{"x":528.1848169877047,"y":330.88140188455327,"color":"c"},{"x":480.7737940317566,"y":375.4227800683542,"color":"c"},{"x":490.72252911063663,"y":355.6388357650958,"color":"c"},{"x":499.47915248683717,"y":350.39009281708604,"color":"c"},{"x":467.68844352527594,"y":382.9688365097534,"color":"c"},{"x":520.8952321948511,"y":360.6788644015381,"color":"c"},{"x":494.76890484938923,"y":377.9863838723098,"color":"c"},{"x":469.3522493144796,"y":395.35661284844537,"color":"c"},{"x":455.79433208506873,"y":369.174285176942,"color":"c"},{"x":461.4785038167495,"y":418.30456615323885,"color":"c"},{"x":470.87630986299934,"y":476.9644838121286,"color":"c"},{"x":494.945536566455,"y":407.04783727118104,"color":"c"},{"x":475.8522526696458,"y":407.6527838518134,"color":"c"},{"x":465.50362466435115,"y":443.7731461417249,"color":"c"},{"x":462.7015177921201,"y":392.02152305594876,"color":"c"},{"x":456.92387862841235,"y":421.87987329647456,"color":"c"},{"x":435.3662067949573,"y":490.78209301667846,"color":"c"},{"x":397.8872069262461,"y":480.57284719941066,"color":"c"},{"x":459.86063511498037,"y":503.8026310431189,"color":"c"},{"x":414.17913348245804,"y":415.48724790106405,"color":"c"},{"x":465.67639207738756,"y":448.0673286168848,"color":"c"},{"x":459.2414117922678,"y":465.841260523383,"color":"c"},{"x":496.70366387398576,"y":437.1866170272341,"color":"c"},{"x":434.129204250308,"y":457.1767386432445,"color":"c"},{"x":454.43914382973435,"y":417.72028882625494,"color":"c"},{"x":479.04372901023964,"y":426.9736045229064,"color":"c"},{"x":496.0902244423522,"y":485.22398568726055,"color":"c"},{"x":427.9816978612212,"y":413.02638565421336,"color":"c"},{"x":445.2619819188208,"y":386.35203046309084,"color":"c"},{"x":460.75432416140205,"y":363.38622758381234,"color":"c"},{"x":491.6480049919749,"y":418.92859119821605,"color":"c"},{"x":501.49035778173135,"y":425.54795399440593,"color":"c"},{"x":485.2310133198505,"y":435.21548778323904,"color":"c"},{"x":507.17109291688286,"y":378.25505679445905,"color":"c"},{"x":507.478329249884,"y":389.4676966470116,"color":"c"},{"x":513.6500286338779,"y":399.0620304470418,"color":"c"},{"x":506.33377137902556,"y":392.32679340567677,"color":"c"},{"x":498.9704880098606,"y":329.52600209453084,"color":"c"},{"x":542.9093619419513,"y":359.6565887112373,"color":"c"},{"x":474.2013412195697,"y":355.814429527568,"color":"c"},{"x":565.4678321924439,"y":345.64936805944,"color":"c"},{"x":564.0104702249978,"y":325.49445490989064,"color":"c"},{"x":531.5666804592628,"y":308.85260742261283,"color":"c"},{"x":529.8598599218475,"y":332.0771933663216,"color":"c"},{"x":542.7579371275497,"y":342.8289897669813,"color":"c"},{"x":511.0311879069878,"y":309.8498359385054,"color":"c"},{"x":523.077540161587,"y":310.44631993294877,"color":"c"},{"x":554.737230535086,"y":305.53117938306946,"color":"c"},{"x":569.166887801481,"y":312.9461666947981,"color":"c"},{"x":544.8291990720902,"y":310.1197094796173,"color":"c"},{"x":557.8758600303396,"y":253.5366868025329,"color":"c"},{"x":526.5028136784629,"y":303.9949175500105,"color":"c"},{"x":565.820408053649,"y":291.45197470546356,"color":"c"},{"x":528.8872772371383,"y":270.36922684992146,"color":"c"},{"x":568.7911341252226,"y":253.81321659724748,"color":"c"},{"x":559.3570189596658,"y":273.2183049173923,"color":"c"},{"x":546.2542967988675,"y":226.14998284357642,"color":"c"},{"x":575.0849154370387,"y":212.51127961365893,"color":"c"},{"x":521.1786171586277,"y":216.1909966073252,"color":"c"},{"x":572.7198649879995,"y":257.4414332986176,"color":"c"},{"x":591.7609508805396,"y":270.30310710004414,"color":"c"},{"x":589.5815036497823,"y":272.7988195877579,"color":"c"},{"x":589.1366974685822,"y":203.1559269738019,"color":"c"},{"x":602.7493127313568,"y":216.94800394787433,"color":"c"},{"x":618.2238506934202,"y":194.2097034739176,"color":"c"},{"x":659.8910965350365,"y":184.23802550032997,"color":"c"},{"x":620.5399025549697,"y":182.4670076045092,"color":"c"},{"x":604.5248808576438,"y":160.10209098518465,"color":"c"},{"x":621.1665628810997,"y":226.10051785697192,"color":"c"},{"x":617.6470151120507,"y":180.1868189109082,"color":"c"},{"x":633.9749048150802,"y":167.86378608816722,"color":"c"},{"x":607.0084061219152,"y":146.6408365852988,"color":"c"},{"x":689.0760110596948,"y":182.29713944044943,"color":"c"},{"x":586.4401575965124,"y":123.4667295057634,"color":"c"},{"x":628.8457811148546,"y":186.78679997754887,"color":"c"},{"x":660.2675442358519,"y":125.66111323977327,"color":"c"},{"x":657.8186891203299,"y":120.82145274202185,"color":"c"},{"x":645.373483921615,"y":202.78075629167506,"color":"c"},{"x":629.6624082838542,"y":154.830735495493,"color":"c"},{"x":617.0512224642163,"y":167.16997419646094,"color":"c"},{"x":664.7469976950482,"y":105.11538217020643,"color":"c"},{"x":667.1132016650829,"y":115.9400036503767,"color":"c"},{"x":650.5873421065554,"y":183.15403007916245,"color":"c"},{"x":670.8100750840186,"y":135.33123888970078,"color":"c"},{"x":654.093304548451,"y":115.77607006120672,"color":"c"},{"x":635.3438575102414,"y":121.21867526537307,"color":"c"},{"x":667.3902626120689,"y":113.53342508221255,"color":"c"},{"x":668.5578801120122,"y":114.01929045833026,"color":"c"},{"x":654.2489869485998,"y":84.23714301425713,"color":"c"},{"x":629.1245575318263,"y":92.70622622458092,"color":"c"},{"x":665.220889265868,"y":148.54816671456086,"color":"c"},{"x":693.6357152487309,"y":104.53961613664973,"color":"c"},{"x":662.5527403651254,"y":137.52528221525836,"color":"c"},{"x":680.6701819991051,"y":499.2191385883926,"color":"c"},{"x":668.5187738510871,"y":442.3674378656026,"color":"c"},{"x":701.4351403663626,"y":411.1312510777191,"color":"c"},{"x":675.404500388924,"y":450.3942897493923,"color":"c"},{"x":669.181387415178,"y":404.42642402897957,"color":"c"},{"x":672.4961766464096,"y":407.18017959183413,"color":"c"},{"x":652.7609653582075,"y":397.2931791990427,"color":"c"},{"x":631.8993643259021,"y":415.87331861119844,"color":"c"},{"x":608.1818743956801,"y":352.7764777855862,"color":"c"},{"x":646.5852277293255,"y":388.51762645026645,"color":"c"},{"x":598.3207917588128,"y":368.8053987527731,"color":"c"},{"x":600.5108079030185,"y":372.5795834740184,"color":"c"},{"x":581.6380197109663,"y":396.2906916352676,"color":"c"},{"x":603.501048345608,"y":347.92805369699033,"color":"c"},{"x":640.3655294254934,"y":363.2304740790748,"color":"c"},{"x":571.8596478038464,"y":335.90829713219324,"color":"c"},{"x":602.4280104710934,"y":340.71745626599443,"color":"c"},{"x":569.8530033313269,"y":326.0534650877226,"color":"c"},{"x":582.1031536212658,"y":360.57874288802964,"color":"c"},{"x":566.3878003712833,"y":307.6241341656249,"color":"c"},{"x":609.3189139779736,"y":338.73220772340756,"color":"c"},{"x":589.202108919072,"y":355.10472898784894,"color":"c"},{"x":640.1360538029214,"y":321.69030611260075,"color":"c"},{"x":605.8169355242009,"y":256.23853473988964,"color":"c"},{"x":572.9325853664691,"y":341.2683693191053,"color":"c"},{"x":571.6229641936767,"y":378.0162721274203,"color":"c"},{"x":602.368995438124,"y":349.99544997065436,"color":"c"},{"x":615.49078396887,"y":370.0723492395482,"color":"c"},{"x":627.5375664786197,"y":353.27203732067557,"color":"c"},{"x":592.7513624950282,"y":370.45190269640784,"color":"c"},{"x":621.2797220875127,"y":368.7739945434405,"color":"c"},{"x":626.8998862686369,"y":418.09203161544644,"color":"c"},{"x":684.7765831730536,"y":434.37396230719224,"color":"c"},{"x":694.9385763140219,"y":401.8818451567789,"color":"c"},{"x":702.3403305976914,"y":433.29854249363655,"color":"c"},{"x":697.6024721916438,"y":428.47780502363383,"color":"c"},{"x":712.8644829684854,"y":471.54923970668784,"color":"c"},{"x":672.8525930612143,"y":472.3302000262413,"color":"c"},{"x":668.728788669287,"y":450.195864808266,"color":"c"},{"x":714.8143497977031,"y":437.739931354434,"color":"c"},{"x":734.4625138717017,"y":440.93625054814595,"color":"c"},{"x":685.5497638348506,"y":444.3560542955327,"color":"c"},{"x":702.2808719952906,"y":441.32358524127943,"color":"c"},{"x":626.0731180037162,"y":420.7217205984093,"color":"c"},{"x":738.1692491001612,"y":500.9381986779846,"color":"c"},{"x":687.9276189509806,"y":449.7528813495286,"color":"c"},{"x":681.201759525986,"y":457.49011632220737,"color":"c"},{"x":718.6167655050539,"y":463.7773405931692,"color":"c"},{"x":697.7044775785316,"y":472.1194141011398,"color":"c"},{"x":728.1056036232734,"y":485.45925388330716,"color":"c"},{"x":667.7822301798533,"y":373.2057847346707,"color":"c"},{"x":669.6541332726584,"y":394.30357655922677,"color":"c"},{"x":583.2090555067922,"y":295.64296439327006,"color":"c"},{"x":590.551737181189,"y":335.44038276040783,"color":"c"},{"x":570.0218348133275,"y":335.1010996096327,"color":"c"},{"x":593.053265261351,"y":333.39658653121694,"color":"c"},{"x":580.6033203935137,"y":337.34861628975824,"color":"c"},{"x":589.1169834244539,"y":272.4743074873297,"color":"c"},{"x":642.5128002694997,"y":305.348065568268,"color":"c"},{"x":566.2336208019502,"y":325.26466190853046,"color":"c"},{"x":557.2991407246816,"y":305.2263205678946,"color":"c"},{"x":533.0070821916486,"y":359.94069933553527,"color":"c"},{"x":560.6510877650863,"y":317.5827825908194,"color":"c"},{"x":553.198979039776,"y":313.18091537269555,"color":"c"},{"x":593.9871682351854,"y":280.7969449946816,"color":"c"},{"x":546.8014573428231,"y":347.3858157192842,"color":"c"},{"x":599.9936783760621,"y":344.9801325440725,"color":"c"},{"x":570.7943943385854,"y":351.7395638548953,"color":"c"},{"x":619.300587350344,"y":371.1442781253203,"color":"c"},{"x":620.2701984784379,"y":347.4515032116866,"color":"c"},{"x":629.7744270767258,"y":363.5756005610199,"color":"c"},{"x":632.4030101933735,"y":399.6550162617523,"color":"c"},{"x":611.1916711178068,"y":358.4345190064408,"color":"c"},{"x":689.4938398897693,"y":418.625457478445,"color":"c"},{"x":674.195417517591,"y":387.6466944533055,"color":"c"},{"x":625.6817379030283,"y":389.65217384258386,"color":"c"},{"x":647.3406756901242,"y":403.2064899823785,"color":"c"},{"x":626.9142357215001,"y":413.7448173414796,"color":"c"},{"x":639.2140493988306,"y":424.43861362673204,"color":"c"},{"x":647.6364503390931,"y":412.2264552201297,"color":"c"},{"x":667.2195987609005,"y":392.56314919792794,"color":"c"},{"x":673.8171088546636,"y":391.47693207440426,"color":"c"},{"x":662.4536213187585,"y":407.21323026858533,"color":"c"},{"x":668.2808809627668,"y":403.79986040887206,"color":"c"},{"x":665.9744750161409,"y":414.7697180823836,"color":"c"},{"x":678.2657579676697,"y":449.6635776906717,"color":"c"},{"x":691.7876507880853,"y":395.60892339312954,"color":"c"},{"x":707.6957485811603,"y":428.5347710853879,"color":"c"},{"x":673.1453699154915,"y":443.8240222549366,"color":"c"},{"x":714.0194268378282,"y":414.6967833555183,"color":"c"},{"x":708.5537996133326,"y":479.516913006654,"color":"c"},{"x":694.2077564336402,"y":456.4763572084172,"color":"c"},{"x":721.3614720977605,"y":431.630831752895,"color":"c"},{"x":720.4978339520515,"y":461.9969744710297,"color":"c"},{"x":672.968028293773,"y":414.3064597283786,"color":"c"},{"x":675.032961641366,"y":374.91823380936796,"color":"c"},{"x":658.7399336615017,"y":394.0407454007417,"color":"c"},{"x":721.2935541338234,"y":415.63909546756366,"color":"c"},{"x":670.7059648221442,"y":369.7481457859999,"color":"c"},{"x":632.7130945791139,"y":351.7938549083668,"color":"c"},{"x":673.0222251757081,"y":376.3528856831154,"color":"c"},{"x":656.6444514478787,"y":356.0927218313154,"color":"c"},{"x":660.521368927053,"y":381.28875454281865,"color":"c"},{"x":622.0121321204604,"y":337.85598426861714,"color":"c"},{"x":625.9913803588748,"y":297.2290093504306,"color":"c"},{"x":663.4939664705415,"y":362.1100926852803,"color":"c"},{"x":605.2671052424878,"y":348.6403393141484,"color":"c"},{"x":607.5813342812537,"y":322.71147400522335,"color":"c"},{"x":604.9892890922599,"y":325.18465833555155,"color":"c"},{"x":579.6816732698092,"y":327.4406627167685,"color":"c"},{"x":628.9379770016586,"y":318.44785628813275,"color":"c"},{"x":619.0470978180538,"y":319.7931768156321,"color":"c"},{"x":591.2618691834803,"y":297.5289951735459,"color":"c"},{"x":577.6642331203132,"y":289.3415260091315,"color":"c"},{"x":620.0930960782086,"y":355.213937262729,"color":"c"},{"x":453.26544707850235,"y":139.39140818758307,"color":"c"},{"x":468.1579828327265,"y":158.04250480730713,"color":"c"},{"x":449.2399118209622,"y":182.88110917542957,"color":"c"},{"x":442.4748448045073,"y":77.42571565866547,"color":"c"},{"x":432.1910823970078,"y":140.19017321040263,"color":"c"},{"x":449.42738448223577,"y":165.1979549742935,"color":"c"},{"x":444.2168837803786,"y":124.59498279232156,"color":"c"},{"x":439.53371292717765,"y":118.6024717665415,"color":"c"},{"x":456.95821165391675,"y":156.92764914672205,"color":"c"},{"x":373.1954827456354,"y":118.32980754915076,"color":"c"},{"x":384.9406226402532,"y":87.82696730539169,"color":"c"},{"x":413.9609445258041,"y":78.60547424605721,"color":"c"},{"x":407.45521404428246,"y":100.0652555275073,"color":"c"},{"x":398.35707936379873,"y":115.16900201928837,"color":"c"},{"x":389.15944652291364,"y":138.61636564168066,"color":"c"},{"x":364.75805387500236,"y":91.51253251192065,"color":"c"},{"x":412.4113611860466,"y":112.54110702539316,"color":"c"},{"x":450.6795815965011,"y":80.533706033214,"color":"c"},{"x":419.17196492052705,"y":144.30399010741013,"color":"c"},{"x":445.3786941044996,"y":149.17247720374678,"color":"c"},{"x":399.1701373713938,"y":150.55229093997127,"color":"c"},{"x":508.5514512810799,"y":139.07827601491584,"color":"c"},{"x":472.2877794822288,"y":159.78878624384384,"color":"c"},{"x":412.02978343606406,"y":149.59945360220394,"color":"c"},{"x":439.50908293848863,"y":197.11491976030675,"color":"c"},{"x":487.2466736004081,"y":218.85427699855,"color":"c"},{"x":467.3157418672811,"y":144.95162397075944,"color":"c"},{"x":498.7457808006172,"y":207.40229275062205,"color":"c"},{"x":440.71433797066146,"y":232.08325705439046,"color":"c"},{"x":531.4362433391384,"y":246.5365878037267,"color":"c"},{"x":528.4173047892735,"y":203.13732372053636,"color":"c"},{"x":561.6796814550663,"y":253.00530937597875,"color":"c"},{"x":519.898783297621,"y":225.72451635409055,"color":"c"},{"x":537.9484378010504,"y":280.8921994279183,"color":"c"},{"x":493.39899592996994,"y":208.22085365651162,"color":"c"},{"x":546.6597577805891,"y":267.5282149066524,"color":"c"},{"x":565.689204715503,"y":271.2427568390317,"color":"c"},{"x":562.3585767942187,"y":235.72319823186706,"color":"c"},{"x":511.8346265109842,"y":267.4848357688239,"color":"c"},{"x":495.3702971704553,"y":242.1381266002652,"color":"c"},{"x":537.092480785199,"y":240.40472653076455,"color":"c"},{"x":534.610415622595,"y":218.8329135515496,"color":"c"},{"x":582.6848816007475,"y":208.77958330848224,"color":"c"},{"x":498.2534106324377,"y":195.02083549929404,"color":"c"},{"x":503.36661471084983,"y":253.46204083533715,"color":"c"},{"x":508.1961497650895,"y":192.09874750722093,"color":"c"},{"x":521.2249800623485,"y":187.2703924613399,"color":"c"},{"x":504.252378514333,"y":180.6634246507072,"color":"c"},{"x":537.0632321359,"y":217.3782149962562,"color":"c"},{"x":499.2268932231835,"y":206.36527390733102,"color":"c"},{"x":515.6487278785916,"y":145.4584199721548,"color":"c"},{"x":534.9957644686299,"y":143.81012292789075,"color":"c"},{"x":523.8146298772397,"y":168.76517421311473,"color":"c"},{"x":473.0062918029751,"y":183.5903663769513,"color":"c"},{"x":463.22481920581004,"y":188.2307302462064,"color":"c"},{"x":479.067771786172,"y":175.73925974908002,"color":"c"},{"x":447.4246398499351,"y":165.4104397181212,"color":"c"},{"x":407.31994568168295,"y":198.83273319505918,"color":"c"},{"x":474.8386051000543,"y":171.6407891380859,"color":"c"},{"x":469.0657039898777,"y":176.70816122851522,"color":"c"},{"x":531.7774412833417,"y":123.96292188239056,"color":"c"},{"x":481.26735101596097,"y":123.15120812116277,"color":"c"},{"x":431.8357174350589,"y":127.59102638136659,"color":"c"},{"x":440.39356956578996,"y":124.86484157780177,"color":"c"},{"x":427.4566223687767,"y":157.63899307176905,"color":"c"},{"x":430.1228970343582,"y":161.1568399748657,"color":"c"},{"x":472.30301833631955,"y":131.5399586478672,"color":"c"},{"x":517.850655119189,"y":130.48705330835884,"color":"c"},{"x":428.09940107853856,"y":131.0176205619269,"color":"c"},{"x":443.81501530968467,"y":172.78889371717895,"color":"c"},{"x":446.9785943497363,"y":142.9975730281015,"color":"c"},{"x":451.73043923387667,"y":138.26403227839785,"color":"c"},{"x":403.85632696468997,"y":81.97775973462427,"color":"c"},{"x":394.6400369133217,"y":144.695424459945,"color":"c"},{"x":401.4878391245231,"y":123.02954842185056,"color":"c"},{"x":426.7449818567191,"y":117.96144645294476,"color":"c"},{"x":398.3029990552336,"y":137.8729162339742,"color":"c"},{"x":389.3512638784104,"y":140.99679139896858,"color":"c"},{"x":418.043265995702,"y":80.06776082362609,"color":"c"},{"x":399.08046593135583,"y":126.42550547187989,"color":"c"},{"x":403.39412543466557,"y":77.90605531931703,"color":"c"},{"x":346.89961063583223,"y":134.3727784775425,"color":"c"},{"x":425.63241873047485,"y":144.6446803137643,"color":"c"},{"x":454.08347994194577,"y":181.23181169581312,"color":"c"},{"x":430.11906722500686,"y":194.07618844950514,"color":"c"},{"x":462.5974248015626,"y":162.5931544631983,"color":"c"},{"x":461.5521114185617,"y":194.21542151546652,"color":"c"},{"x":501.8952380366118,"y":181.99453174189546,"color":"c"},{"x":452.95047700903416,"y":191.57008522949008,"color":"c"},{"x":481.56156804212105,"y":213.38924099172453,"color":"c"},{"x":513.084792520273,"y":205.98892483583302,"color":"c"},{"x":468.28532278589296,"y":191.9573483551153,"color":"c"},{"x":519.3134134655231,"y":209.97323476893928,"color":"c"},{"x":535.5758878730244,"y":247.44837978684114,"color":"c"},{"x":495.8296379664723,"y":224.90363945396342,"color":"c"},{"x":516.4488686974585,"y":240.7106358970998,"color":"c"},{"x":532.8392689859462,"y":227.29344798163248,"color":"c"},{"x":524.0858215153725,"y":276.76206975698346,"color":"c"},{"x":528.6874371180472,"y":191.0040823994466,"color":"c"},{"x":478.2500354151726,"y":206.75402658248083,"color":"c"},{"x":478.6577622185261,"y":253.34283836417976,"color":"c"},{"x":532.1694684112734,"y":205.7147030277905,"color":"c"},{"x":547.0316800843354,"y":239.2930578557416,"color":"c"},{"x":528.1841476751625,"y":208.2510893874731,"color":"c"},{"x":566.0546219697719,"y":250.12407658168237,"color":"c"},{"x":543.7757099314047,"y":274.2149896992701,"color":"c"},{"x":576.4256048855995,"y":406.72430596102765,"color":"d"},{"x":569.1697265043372,"y":454.8971391363724,"color":"d"},{"x":603.4561010737584,"y":438.1662829436218,"color":"d"},{"x":561.274672155612,"y":410.92375111062233,"color":"d"},{"x":544.9725139321885,"y":461.1422365295764,"color":"d"},{"x":538.8928553078124,"y":431.2465932522715,"color":"d"},{"x":566.0750271914819,"y":436.1308509416606,"color":"d"},{"x":552.1046541810276,"y":406.021986478811,"color":"d"},{"x":547.8817011036111,"y":461.4659787508667,"color":"d"},{"x":583.32435078472,"y":433.02941214809977,"color":"d"},{"x":568.5329174099331,"y":441.8646667119449,"color":"d"},{"x":560.6311096225339,"y":332.4321481553478,"color":"d"},{"x":569.6454257582143,"y":485.91569093890337,"color":"d"},{"x":594.4724219579766,"y":433.8620543412229,"color":"d"},{"x":569.0602076351098,"y":461.2686725119215,"color":"d"},{"x":606.9972696378408,"y":426.79138046289535,"color":"d"},{"x":606.9861106764243,"y":477.6040138617209,"color":"d"},{"x":630.0717211989925,"y":370.2550467083091,"color":"d"},{"x":596.3171266706368,"y":417.03504744080607,"color":"d"},{"x":572.8534730166174,"y":413.3207095565833,"color":"d"},{"x":559.1985291605374,"y":468.9371549497547,"color":"d"},{"x":561.0423469454726,"y":405.85437273978346,"color":"d"},{"x":589.3598489967766,"y":450.90204841554123,"color":"d"},{"x":566.4325234098337,"y":451.5379678584727,"color":"d"},{"x":571.7864506710462,"y":489.3453532998407,"color":"d"},{"x":549.6381470787414,"y":371.9099735098613,"color":"d"},{"x":567.0620151093461,"y":489.02257333108,"color":"d"},{"x":592.4664596032612,"y":436.39248952612866,"color":"d"},{"x":540.5676316845078,"y":453.5189172863761,"color":"d"},{"x":559.2952508483481,"y":497.80066637623884,"color":"d"},{"x":522.7982960757157,"y":461.3499952740101,"color":"d"},{"x":523.5418502272894,"y":457.4373696942473,"color":"d"},{"x":554.8398036882303,"y":397.3696411916812,"color":"d"},{"x":558.3423792321774,"y":454.2519301597431,"color":"d"},{"x":526.6583236681745,"y":402.42890619547154,"color":"d"},{"x":595.159999591891,"y":437.68960069108766,"color":"d"},{"x":572.8145291981555,"y":466.10371876494196,"color":"d"},{"x":587.1865367097538,"y":443.58136137822316,"color":"d"},{"x":552.3594045555409,"y":452.03810652204766,"color":"d"},{"x":543.5376230273654,"y":421.49347255811426,"color":"d"},{"x":526.4897031704755,"y":469.21584484421044,"color":"d"},{"x":530.4730697563124,"y":421.7123542318269,"color":"d"},{"x":550.8486199132133,"y":465.72768692878407,"color":"d"},{"x":595.5955259822734,"y":487.8866131517279,"color":"d"},{"x":611.2201073944809,"y":418.3221234537514,"color":"d"},{"x":579.5446669835429,"y":502.1366429278812,"color":"d"},{"x":589.0613935526139,"y":449.8876104995321,"color":"d"},{"x":609.7087511035606,"y":458.034712053747,"color":"d"},{"x":594.6196315051994,"y":497.52488550432895,"color":"d"},{"x":635.7525405159869,"y":497.62406090267433,"color":"d"},{"x":619.6146415782927,"y":415.67101398125925,"color":"d"},{"x":615.9948033048929,"y":459.3892413468656,"color":"d"},{"x":577.1345550936834,"y":431.07866313293005,"color":"d"},{"x":555.6639328605551,"y":374.32081225886,"color":"d"},{"x":558.9231898912944,"y":431.99225680502553,"color":"d"},{"x":549.5297739040535,"y":413.504132950254,"color":"d"},{"x":609.3882149811525,"y":391.7555796707541,"color":"d"},{"x":557.7667585072638,"y":424.6887936776194,"color":"d"},{"x":581.086147886036,"y":345.4279546682184,"color":"d"},{"x":538.2967655850142,"y":376.47879922808,"color":"d"},{"x":573.5121211987524,"y":404.1821436238852,"color":"d"},{"x":582.11057347785,"y":427.6591347326869,"color":"d"},{"x":530.4333525516823,"y":398.0851199914736,"color":"d"},{"x":589.1517125129027,"y":350.5126216018657,"color":"d"},{"x":578.0905919829571,"y":417.9377326629393,"color":"d"},{"x":571.6106063767621,"y":380.44813839943345,"color":"d"},{"x":538.8295799388502,"y":360.357449448789,"color":"d"},{"x":559.0375927567625,"y":359.6666093714268,"color":"d"},{"x":586.7966462012323,"y":379.2314140638025,"color":"d"},{"x":552.3844112022489,"y":443.2233850887899,"color":"d"},{"x":561.0635925908186,"y":472.44795797632486,"color":"d"},{"x":567.7888538693097,"y":429.7771933571159,"color":"d"},{"x":542.2647299290804,"y":391.04695176296343,"color":"d"},{"x":589.3161362742948,"y":397.661964265748,"color":"d"},{"x":591.5558300036812,"y":420.94522092485033,"color":"d"},{"x":549.6797272942443,"y":382.45286227597944,"color":"d"},{"x":569.5485009712271,"y":421.30531266308,"color":"d"},{"x":562.4486352477134,"y":443.2288693779926,"color":"d"},{"x":538.7139508486232,"y":449.3479892139162,"color":"d"},{"x":570.9268763553341,"y":461.0013598618499,"color":"d"},{"x":584.0599673957355,"y":439.2044181369261,"color":"d"},{"x":403.8649119287386,"y":237.23798665633774,"color":"d"},{"x":455.22343455619085,"y":265.9226442081896,"color":"d"},{"x":444.7192210503269,"y":316.08503597315797,"color":"d"},{"x":499.0357626043757,"y":305.9684829630682,"color":"d"},{"x":484.7569007442362,"y":260.52081884527126,"color":"d"},{"x":505.6153584602494,"y":274.7454817575715,"color":"d"},{"x":496.20102674595495,"y":293.86450842357397,"color":"d"},{"x":449.6212366210763,"y":312.3792691305152,"color":"d"},{"x":464.0156252880179,"y":288.93027649745045,"color":"d"},{"x":487.9054908254848,"y":253.38015324248113,"color":"d"},{"x":481.84824124823604,"y":253.49254530086117,"color":"d"},{"x":480.66350335853497,"y":261.14632009476543,"color":"d"},{"x":442.4261481918385,"y":276.02520723771636,"color":"d"},{"x":450.43468881425343,"y":328.48030550991086,"color":"d"},{"x":459.5195802432914,"y":258.6109028856554,"color":"d"},{"x":383.06289584732696,"y":287.37123683207267,"color":"d"},{"x":491.4915876561313,"y":251.83864538215198,"color":"d"},{"x":428.558141792574,"y":320.7710378647024,"color":"d"},{"x":475.2855482237126,"y":196.4226665803497,"color":"d"},{"x":407.0581911511346,"y":247.84827763545564,"color":"d"},{"x":446.83393054238053,"y":313.915852815313,"color":"d"},{"x":409.2394269864538,"y":358.13767245281855,"color":"d"},{"x":402.60427295003814,"y":312.42228197475515,"color":"d"},{"x":449.6249215590749,"y":282.4403376081726,"color":"d"},{"x":432.76364972775593,"y":247.43693790224097,"color":"d"},{"x":478.25261462840405,"y":244.560114429984,"color":"d"},{"x":436.0853945451895,"y":305.37847495499,"color":"d"},{"x":432.63362578637737,"y":316.4904495265646,"color":"d"},{"x":427.6290563538044,"y":304.9284754318493,"color":"d"},{"x":467.4872498798013,"y":281.4213789007524,"color":"d"},{"x":496.7790841349489,"y":278.56450099158025,"color":"d"},{"x":443.07469820135617,"y":285.5495029471724,"color":"d"},{"x":424.4706468373067,"y":300.98850169734806,"color":"d"},{"x":452.96199489849386,"y":273.6483962470779,"color":"d"},{"x":447.02487603714053,"y":283.578832840421,"color":"d"},{"x":483.5370848544567,"y":273.0807558256119,"color":"d"},{"x":469.61998938903406,"y":253.4393935366558,"color":"d"},{"x":479.64935304622117,"y":322.0958005147596,"color":"d"},{"x":430.4075198125109,"y":291.02965024181174,"color":"d"},{"x":428.50181694023934,"y":271.0813424241589,"color":"d"},{"x":431.8533618760241,"y":265.81755223704647,"color":"d"},{"x":454.1177031716002,"y":293.5885729454809,"color":"d"},{"x":471.6585039198096,"y":342.3777301346647,"color":"d"},{"x":430.13861173682096,"y":314.14180465091334,"color":"d"},{"x":405.8554186285065,"y":291.60514447036155,"color":"d"},{"x":479.0033189045956,"y":277.09302692336746,"color":"d"},{"x":476.4688938214813,"y":333.9152752263378,"color":"d"},{"x":376.4764479790935,"y":362.2540635905262,"color":"d"},{"x":458.5673932123906,"y":326.52938604493113,"color":"d"},{"x":506.52849854987664,"y":317.26605703169207,"color":"d"},{"x":443.3473468455739,"y":294.393386758064,"color":"d"},{"x":454.62369878866417,"y":292.7432108181225,"color":"d"},{"x":457.2589061418773,"y":273.71586597150497,"color":"d"},{"x":482.7400033705055,"y":266.53925220340875,"color":"d"},{"x":415.8002143011708,"y":242.79663728546677,"color":"d"},{"x":478.68300953485704,"y":265.653562307237,"color":"d"},{"x":417.666269338468,"y":293.4442630250717,"color":"d"},{"x":433.96619749607845,"y":239.14337216874458,"color":"d"},{"x":452.68671638551604,"y":220.91099520207592,"color":"d"},{"x":461.7647358173764,"y":224.02301149010805,"color":"d"},{"x":440.58156903024485,"y":260.2790999350999,"color":"d"},{"x":478.698799160957,"y":274.86887928130767,"color":"d"},{"x":527.0038793918253,"y":279.5274012898424,"color":"d"},{"x":472.27069433804485,"y":197.75165842932842,"color":"d"},{"x":501.7189483849491,"y":255.85947937178207,"color":"d"},{"x":447.0275560455038,"y":259.7143942550015,"color":"d"},{"x":473.62831033453165,"y":295.18545404364517,"color":"d"},{"x":504.9019840898817,"y":254.60023556903693,"color":"d"},{"x":488.72260412526805,"y":283.8528044001453,"color":"d"},{"x":475.6884993647984,"y":252.500052950564,"color":"d"},{"x":469.3334060204882,"y":228.16127712681578,"color":"d"},{"x":423.50344093486285,"y":278.1495211215875,"color":"d"},{"x":468.68996120275034,"y":264.12325282939184,"color":"d"},{"x":460.7833224537716,"y":301.68991676154246,"color":"d"},{"x":430.2206272665037,"y":252.28544227836375,"color":"d"},{"x":441.8432171163307,"y":256.71062494022874,"color":"d"},{"x":456.8297818766328,"y":305.3250593330951,"color":"d"},{"x":425.8137972435242,"y":284.73131238507915,"color":"d"},{"x":470.3517889382618,"y":266.8233197481658,"color":"d"},{"x":455.2092066598729,"y":205.69376474658418,"color":"d"},{"x":475.47493799782825,"y":243.94426071611554,"color":"d"},{"x":436.68572298574685,"y":284.36206949776727,"color":"d"},{"x":483.3140138771634,"y":236.31255102801953,"color":"d"},{"x":378.21444048811475,"y":245.6311261190084,"color":"d"},{"x":445.77606739320805,"y":262.1969745221609,"color":"d"},{"x":445.5773704991299,"y":262.35501008038676,"color":"d"},{"x":444.338420880926,"y":259.2779765969808,"color":"d"},{"x":470.07391670198524,"y":245.26178694833746,"color":"d"},{"x":436.22086714393265,"y":219.29841141586735,"color":"d"},{"x":443.5024476461211,"y":256.79426778659587,"color":"d"},{"x":498.9078551281955,"y":182.32058576560223,"color":"d"},{"x":427.72398744940693,"y":231.4726080667536,"color":"d"},{"x":652.2606813016836,"y":303.4729651550625,"color":"d"},{"x":627.117490308325,"y":271.55807115169057,"color":"d"},{"x":683.48319723772,"y":278.8911580687787,"color":"d"},{"x":644.5915816726803,"y":296.22143648295986,"color":"d"},{"x":667.6267963650821,"y":263.4351149069438,"color":"d"},{"x":675.6509921834656,"y":251.25631944920286,"color":"d"},{"x":672.088146100973,"y":300.9655427472514,"color":"d"},{"x":697.4549862276044,"y":249.17038009049054,"color":"d"},{"x":674.046002793006,"y":209.14923731611634,"color":"d"},{"x":649.7917141608749,"y":252.57078255595155,"color":"d"},{"x":663.8794974090601,"y":253.29807726401458,"color":"d"},{"x":642.231136700631,"y":219.09409319414635,"color":"d"},{"x":559.9366113036392,"y":278.99396389283686,"color":"d"},{"x":669.9106116118387,"y":226.0270205654757,"color":"d"},{"x":603.9011033414167,"y":244.1596377248937,"color":"d"},{"x":622.0628184593397,"y":237.25188139849303,"color":"d"},{"x":659.4879898224729,"y":223.57273707056618,"color":"d"},{"x":637.743552245057,"y":261.99949541813925,"color":"d"},{"x":655.9411023001879,"y":263.95846329933494,"color":"d"},{"x":606.406047597491,"y":254.84381341456034,"color":"d"},{"x":682.6080284599786,"y":265.59699715588795,"color":"d"},{"x":645.3815765495037,"y":311.70679453648336,"color":"d"},{"x":670.4724287017865,"y":325.9083966326184,"color":"d"},{"x":677.1235723463243,"y":269.0066041537921,"color":"d"},{"x":674.8212817569699,"y":288.6958636962751,"color":"d"},{"x":672.2947025913527,"y":282.0571696898615,"color":"d"},{"x":667.3737487251308,"y":287.3453342961718,"color":"d"},{"x":676.6226077870102,"y":324.2635853070984,"color":"d"},{"x":698.6403839138411,"y":311.8576030085362,"color":"d"},{"x":689.7294923679153,"y":313.3542819434058,"color":"d"},{"x":665.7545452198825,"y":269.88092981729864,"color":"d"},{"x":650.0756809303143,"y":324.52840183537785,"color":"d"},{"x":720.2632937361601,"y":282.7867089491146,"color":"d"},{"x":697.6072021181266,"y":264.4186187070686,"color":"d"},{"x":654.9825175847145,"y":243.90037334381725,"color":"d"},{"x":679.6844906710307,"y":258.1439368879644,"color":"d"},{"x":668.298368953404,"y":244.05205244024557,"color":"d"},{"x":645.0064678031918,"y":265.4357911442088,"color":"d"},{"x":706.8679412003996,"y":232.66436168667946,"color":"d"},{"x":711.595924064511,"y":257.04666886448035,"color":"d"},{"x":651.2180075303835,"y":181.36680171991657,"color":"d"},{"x":720.5491951870043,"y":307.88664953221564,"color":"d"},{"x":670.9426043050063,"y":244.35100676786178,"color":"d"},{"x":647.8986505568981,"y":264.68364897939466,"color":"d"},{"x":629.2426604832276,"y":280.23413602001466,"color":"d"},{"x":639.2805489830405,"y":262.26749520238593,"color":"d"},{"x":656.4447725317973,"y":197.87655349948534,"color":"d"},{"x":701.4399088546348,"y":300.3525935477818,"color":"d"},{"x":664.9076458500344,"y":271.5511583392995,"color":"d"},{"x":670.8996108796605,"y":287.26123935747955,"color":"d"},{"x":658.8466195970939,"y":252.8269243221225,"color":"d"},{"x":659.2904820573846,"y":247.2707505917313,"color":"d"},{"x":641.9991157652545,"y":293.6686754353077,"color":"d"},{"x":679.9468502405196,"y":252.23423639028866,"color":"d"},{"x":631.6325253235776,"y":255.95786554742006,"color":"d"},{"x":618.7790547373912,"y":261.662272858141,"color":"d"},{"x":685.9215143659026,"y":249.085573115938,"color":"d"},{"x":675.0904203995728,"y":235.23007252071392,"color":"d"},{"x":662.7222122135025,"y":255.5953503750827,"color":"d"},{"x":641.5936043917585,"y":233.32714820431283,"color":"d"},{"x":649.0051448132406,"y":230.3114523499503,"color":"d"},{"x":677.8248554104473,"y":255.69508574322896,"color":"d"},{"x":673.0906746897822,"y":247.9904180401134,"color":"d"},{"x":693.3034517162638,"y":239.1390999979477,"color":"d"},{"x":680.5133020351201,"y":282.02196889613924,"color":"d"},{"x":689.872527543287,"y":300.69188735356136,"color":"d"},{"x":700.6765392785499,"y":330.91260733692246,"color":"d"},{"x":713.8127098440418,"y":270.4757585891729,"color":"d"},{"x":649.5206928273853,"y":289.2480812007607,"color":"d"},{"x":675.2958812127673,"y":368.19504724097715,"color":"d"},{"x":696.8029722545958,"y":312.72016884695154,"color":"d"},{"x":681.8548354047066,"y":313.04143551478614,"color":"d"},{"x":692.4948306115056,"y":340.9447782277409,"color":"d"},{"x":732.5835293665173,"y":337.1480935093001,"color":"d"},{"x":710.8316005763351,"y":362.45382467565264,"color":"d"},{"x":715.7738726790558,"y":341.809638070604,"color":"d"},{"x":683.6310074820667,"y":337.91599627236553,"color":"d"},{"x":706.6507465934562,"y":321.6351119776434,"color":"d"},{"x":708.5162926871619,"y":241.94426656526173,"color":"d"},{"x":631.7489610099195,"y":279.0235383780199,"color":"d"},{"x":657.0265693162642,"y":280.3492365556536,"color":"d"},{"x":650.253159865128,"y":237.45293086702287,"color":"d"},{"x":675.7314022973821,"y":208.19234207403588,"color":"d"},{"x":685.7159355237836,"y":237.14083917513642,"color":"d"},{"x":650.4061191354938,"y":248.78366931622253,"color":"d"},{"x":695.0615182415435,"y":193.81571822212197,"color":"d"},{"x":704.0268764234838,"y":179.49941126309096,"color":"d"},{"x":660.7471305935793,"y":237.59043498628597,"color":"d"},{"x":648.8745251100122,"y":187.52093157974411,"color":"d"},{"x":616.8072582345311,"y":230.5165836234898,"color":"d"},{"x":680.1908638101672,"y":239.6862826771868,"color":"d"},{"x":685.5143035588692,"y":231.06129500759,"color":"d"},{"x":713.8337708508201,"y":223.92428859144474,"color":"d"},{"x":642.2775319299211,"y":257.99992349763727,"color":"d"},{"x":642.6895301696363,"y":246.70016638834514,"color":"d"},{"x":641.6096876264509,"y":241.17260306792383,"color":"d"},{"x":607.9299574199738,"y":289.0587074585929,"color":"d"},{"x":604.5562796964022,"y":294.45299401302606,"color":"d"},{"x":694.9376301575136,"y":256.193412866736,"color":"d"},{"x":676.5045279587159,"y":275.5906671421401,"color":"d"},{"x":653.2542773225057,"y":291.68893768945054,"color":"d"},{"x":635.3989230397651,"y":274.2981471452225,"color":"d"},{"x":511.0139483960743,"y":133.75832039409448,"color":"d"},{"x":517.4346928379678,"y":111.32107093369473,"color":"d"},{"x":548.7771138698306,"y":120.86853924635722,"color":"d"},{"x":532.7615499824824,"y":119.88700993064481,"color":"d"},{"x":567.3261452692194,"y":121.93585824112677,"color":"d"},{"x":589.7779014189953,"y":157.83594712652604,"color":"d"},{"x":536.9577247679955,"y":195.08186869507307,"color":"d"},{"x":585.207078908883,"y":158.4190245930506,"color":"d"},{"x":592.115347210694,"y":157.21640068824388,"color":"d"},{"x":492.9506118656517,"y":220.27937693368074,"color":"d"},{"x":554.1179114393004,"y":229.58593644538007,"color":"d"},{"x":571.4470598034301,"y":142.84553134840883,"color":"d"},{"x":557.1340056739122,"y":195.24155463950234,"color":"d"},{"x":524.1350238989381,"y":201.78977967864188,"color":"d"},{"x":556.1076365727603,"y":178.3709816928628,"color":"d"},{"x":520.3669881838173,"y":132.32780735556878,"color":"d"},{"x":536.170979805157,"y":150.60443458534752,"color":"d"},{"x":569.9301091675406,"y":140.87443156263856,"color":"d"},{"x":554.2552935098341,"y":59.45744332839979,"color":"d"},{"x":530.0093854897798,"y":16.492579058529486,"color":"d"},{"x":520.2022999679383,"y":135.02036613065678,"color":"d"},{"x":562.8225588861667,"y":53.15958857404223,"color":"d"},{"x":502.7122672256551,"y":107.66622734316763,"color":"d"},{"x":469.9983640414109,"y":91.65971334459277,"color":"d"},{"x":522.6723510937458,"y":111.96557125409822,"color":"d"},{"x":521.1330690603772,"y":90.64626841043145,"color":"d"},{"x":526.536761973485,"y":159.7157329953513,"color":"d"},{"x":512.8974668231353,"y":143.1779716103046,"color":"d"},{"x":479.8615175433782,"y":154.27692335908927,"color":"d"},{"x":526.040603441533,"y":139.9910074712443,"color":"d"},{"x":514.2774301952005,"y":198.77824579329314,"color":"d"},{"x":506.8212747300154,"y":100.86283315216042,"color":"d"},{"x":575.2023044409611,"y":133.53660106178677,"color":"d"},{"x":555.5709824054218,"y":115.90859787779868,"color":"d"},{"x":525.0275762091125,"y":178.11085347531622,"color":"d"},{"x":539.6726657632993,"y":145.5384641102998,"color":"d"},{"x":511.9187751237931,"y":144.1727479858049,"color":"d"},{"x":554.593194921915,"y":114.02726907768363,"color":"d"},{"x":540.6534616790578,"y":148.573414220634,"color":"d"},{"x":524.1927764205767,"y":111.44326767713056,"color":"d"},{"x":547.2242149088302,"y":95.56281889590849,"color":"d"},{"x":549.0391387028859,"y":125.62076151545676,"color":"d"},{"x":568.9913538689941,"y":91.34961704965156,"color":"d"},{"x":520.010053662148,"y":106.35666215599065,"color":"d"},{"x":567.4516604309307,"y":165.5315495071481,"color":"d"},{"x":563.3178468342345,"y":93.66205417159347,"color":"d"},{"x":579.3640158732906,"y":132.74559075038275,"color":"d"},{"x":541.7787053410198,"y":84.4306087052903,"color":"d"},{"x":554.5986227365038,"y":94.45446557922946,"color":"d"},{"x":554.4549834182502,"y":105.96353240182526,"color":"d"},{"x":559.2893995047245,"y":131.7111429760119,"color":"d"},{"x":588.3585715000601,"y":111.40881764574084,"color":"d"},{"x":522.6217817722718,"y":136.5965655996324,"color":"d"},{"x":539.526203136637,"y":173.0562152663996,"color":"d"},{"x":629.6482347667661,"y":130.83756900766917,"color":"d"},{"x":584.4860736403222,"y":152.06420280204276,"color":"d"},{"x":518.2014137745502,"y":165.16958251264748,"color":"d"},{"x":578.6214897420829,"y":178.69528667487793,"color":"d"},{"x":550.7910870088696,"y":129.79080222456957,"color":"d"},{"x":488.6635850770424,"y":113.09091648731226,"color":"d"},{"x":570.4178486386736,"y":130.59243975334516,"color":"d"},{"x":536.2504157763474,"y":177.6557442788258,"color":"d"},{"x":523.5529172203346,"y":200.74103921812736,"color":"d"},{"x":519.0325626597746,"y":141.78986365186864,"color":"d"},{"x":527.0508305323692,"y":131.70434611300198,"color":"d"},{"x":534.4994059022436,"y":90.63635302899519,"color":"d"},{"x":479.5489610933313,"y":149.21426914453895,"color":"d"},{"x":516.1749675366214,"y":118.43401648285794,"color":"d"},{"x":533.1503297381745,"y":106.44507373135326,"color":"d"},{"x":523.7404615357145,"y":124.1739342881636,"color":"d"},{"x":549.0901568981793,"y":170.92419147560412,"color":"d"},{"x":560.5657476596464,"y":110.45211027014977,"color":"d"},{"x":565.575330207103,"y":86.44800798284643,"color":"d"},{"x":552.7166828903897,"y":119.36534913108852,"color":"d"},{"x":516.5255255582339,"y":116.09347431620301,"color":"d"},{"x":576.4850923007615,"y":123.49475018848227,"color":"d"},{"x":542.3870146968983,"y":177.52764134942686,"color":"d"},{"x":533.7175862930935,"y":189.79701096716957,"color":"d"},{"x":578.7367841385679,"y":152.2559352012542,"color":"d"},{"x":541.7439423865156,"y":192.56789269999172,"color":"d"},{"x":546.9665548289494,"y":193.47070775171642,"color":"d"},{"x":556.1775228037552,"y":220.26714584935746,"color":"d"},{"x":536.2732559738453,"y":177.42140794567138,"color":"d"},{"x":545.7314534728886,"y":180.03870154629118,"color":"d"},{"x":556.0159379345039,"y":167.0946287967103,"color":"d"},{"x":514.4377048264885,"y":159.28442254865683,"color":"d"},{"x":518.9474612609243,"y":152.9903031329642,"color":"d"},{"x":553.4572810026937,"y":139.144207293509,"color":"d"},{"x":556.1090883111273,"y":143.58435296197797,"color":"d"},{"x":637.896361011327,"y":126.6208949209364,"color":"d"},{"x":536.3773005982108,"y":98.03276523951831,"color":"d"},{"x":562.494878828449,"y":113.29386230359603,"color":"d"},{"x":548.1653390365288,"y":116.76663286603593,"color":"d"},{"x":593.7593350844041,"y":135.28529354690676,"color":"d"},{"x":580.6804056678095,"y":107.27359342143575,"color":"d"},{"x":551.670364987853,"y":115.67202259772807,"color":"d"},{"x":622.6873300764084,"y":115.65377726035604,"color":"d"},{"x":585.8603169693816,"y":122.5085712626875,"color":"d"},{"x":602.6075813799321,"y":106.03911591793042,"color":"d"},{"x":623.0612393871502,"y":118.61019474402184,"color":"d"},{"x":588.6374112383224,"y":137.54920897891316,"color":"d"},{"x":633.5498729111308,"y":112.1433442992988,"color":"d"},{"x":568.9411747926266,"y":107.24260934861974,"color":"d"},{"x":607.4870875777048,"y":127.50948738659963,"color":"d"},{"x":586.8644814680106,"y":163.24219736058194,"color":"d"},{"x":585.8224624256551,"y":156.33009287262513,"color":"d"},{"x":560.0722548883966,"y":146.34344934559095,"color":"d"},{"x":586.1187303141701,"y":137.08960145016306,"color":"d"},{"x":531.8524950264201,"y":85.79623089555605,"color":"d"},{"x":545.1526849939554,"y":170.45390250701513,"color":"d"},{"x":524.8487638757589,"y":135.92427264079583,"color":"d"},{"x":547.3906722506368,"y":95.12528028188603,"color":"d"},{"x":515.9918302098309,"y":163.62580992695257,"color":"d"},{"x":505.61198510729054,"y":140.72406810971574,"color":"d"},{"x":476.9349061092023,"y":120.32728251584064,"color":"d"},{"x":538.0634941009776,"y":116.71421850745372,"color":"d"},{"x":482.6078038139354,"y":133.46340219967266,"color":"d"},{"x":516.5611734297606,"y":117.044488724139,"color":"d"},{"x":499.622875578498,"y":145.12014281118462,"color":"d"},{"x":488.9491738074647,"y":87.19937037991599,"color":"d"},{"x":502.6410186857117,"y":113.574847587504,"color":"d"},{"x":505.4808502056062,"y":118.56766201562522,"color":"d"},{"x":518.4231855191421,"y":119.29889673151735,"color":"d"},{"x":463.7797520601199,"y":156.40933807127192,"color":"d"},{"x":532.444098881248,"y":134.44428402413854,"color":"d"},{"x":160.84630838327257,"y":252.99157148377924,"color":"b"},{"x":206.3978612439757,"y":237.3590465191537,"color":"b"},{"x":191.0687258694805,"y":278.7807873180011,"color":"b"},{"x":216.50209580780708,"y":254.81541797999782,"color":"b"},{"x":236.03685349268497,"y":215.93251323640266,"color":"b"},{"x":212.38312737591883,"y":229.1902111744381,"color":"b"},{"x":178.25736852353026,"y":248.71836640300012,"color":"b"},{"x":192.38886071435658,"y":257.3890909738145,"color":"b"},{"x":183.42510792358394,"y":286.539371057508,"color":"b"},{"x":230.40496203003838,"y":277.5545085973268,"color":"b"},{"x":174.10025735476745,"y":295.1465382168435,"color":"b"},{"x":226.79554265871934,"y":244.0865383957717,"color":"b"},{"x":189.5563393017366,"y":240.32458863153533,"color":"b"},{"x":200.5194189683375,"y":237.22486004957165,"color":"b"},{"x":230.9607527590085,"y":296.4436593291267,"color":"b"},{"x":211.88388510512362,"y":212.3678285901807,"color":"b"},{"x":192.04753732959165,"y":198.63588034151496,"color":"b"},{"x":200.36826705259307,"y":225.85052118444594,"color":"b"},{"x":196.57868446988823,"y":238.10395249830526,"color":"b"},{"x":164.99981236924506,"y":217.54598614871958,"color":"b"},{"x":185.3615312027772,"y":269.7231178126125,"color":"b"},{"x":158.70892814986723,"y":229.74598068573619,"color":"b"},{"x":199.17775865769403,"y":244.38118113548282,"color":"b"},{"x":216.8865543452881,"y":226.70021392796485,"color":"b"},{"x":259.57019720049107,"y":211.9920389774033,"color":"b"},{"x":256.5001400042687,"y":236.3206777647469,"color":"b"},{"x":215.70235936603464,"y":231.29754759839057,"color":"b"},{"x":185.58570974923904,"y":305.934812021463,"color":"b"},{"x":195.27348154925798,"y":239.70523299991504,"color":"b"},{"x":246.28704240334494,"y":277.89773836021163,"color":"b"},{"x":217.74696110276264,"y":265.61298903981987,"color":"b"},{"x":211.9644575698747,"y":284.9462633392514,"color":"b"},{"x":216.2479982210172,"y":268.91831401189916,"color":"b"},{"x":212.9836864411389,"y":225.65153080850178,"color":"b"},{"x":217.34437565118495,"y":224.8085490307608,"color":"b"},{"x":206.67050983625856,"y":267.1779899688191,"color":"b"},{"x":166.47901732964266,"y":246.37086478078564,"color":"b"},{"x":132.90570240520333,"y":231.73796747305875,"color":"b"},{"x":164.7641640360311,"y":271.118466893529,"color":"b"},{"x":209.17602560880022,"y":238.14353594676777,"color":"b"},{"x":183.25750905927356,"y":237.03743763347478,"color":"b"},{"x":236.72494085206756,"y":220.78921683048475,"color":"b"},{"x":237.68621444534864,"y":201.95083037407943,"color":"b"},{"x":190.32036905751016,"y":213.77364393786752,"color":"b"},{"x":209.92299757134543,"y":220.45824798247367,"color":"b"},{"x":235.29383051802282,"y":249.2800447296333,"color":"b"},{"x":248.5342514750012,"y":259.1261851668918,"color":"b"},{"x":214.90325304312796,"y":263.191361201189,"color":"b"},{"x":261.2188112615385,"y":249.02082701762293,"color":"b"},{"x":254.94845127638987,"y":282.026191537145,"color":"b"},{"x":242.4857955219234,"y":198.58426002485578,"color":"b"},{"x":178.75298691535545,"y":222.56044356290664,"color":"b"},{"x":210.49628862645227,"y":262.9319338209722,"color":"b"},{"x":220.74661062475676,"y":259.9157093760276,"color":"b"},{"x":215.11299449560906,"y":313.97246889208975,"color":"b"},{"x":215.69658373901368,"y":277.9485318689326,"color":"b"},{"x":181.6656980486562,"y":306.5770604849653,"color":"b"},{"x":234.5262329119395,"y":301.5085363728754,"color":"b"},{"x":217.01063176010666,"y":237.45634085369642,"color":"b"},{"x":172.8503374057677,"y":278.14800716677894,"color":"b"},{"x":169.90757224262956,"y":325.2085371865855,"color":"b"},{"x":202.14849442334406,"y":242.4665455670962,"color":"b"},{"x":236.03427066253823,"y":223.3915665950875,"color":"b"},{"x":173.75631410564836,"y":260.92477701903016,"color":"b"},{"x":167.29658439562766,"y":196.7635141890596,"color":"b"},{"x":168.64900818143514,"y":189.53159170433884,"color":"b"},{"x":193.79799337528988,"y":185.1088824419317,"color":"b"},{"x":197.74113880299575,"y":269.1599776048419,"color":"b"},{"x":221.14841971009707,"y":221.5750762289179,"color":"b"},{"x":210.545602402367,"y":253.18255920251812,"color":"b"},{"x":199.83746384840896,"y":239.2575978330932,"color":"b"},{"x":219.83233844130814,"y":245.87851961893787,"color":"b"},{"x":215.39156161134446,"y":306.2086415980998,"color":"b"},{"x":239.25731781576278,"y":283.71016134168906,"color":"b"},{"x":196.7402224876456,"y":255.22901226725665,"color":"b"},{"x":245.59691898209778,"y":248.5254479482989,"color":"b"},{"x":279.89169426578536,"y":218.35697817459504,"color":"b"},{"x":235.58259038983155,"y":235.681422729029,"color":"b"},{"x":232.30477949293643,"y":230.91457891957418,"color":"b"},{"x":240.9170702113248,"y":237.93224556153933,"color":"b"},{"x":218.09766251102417,"y":198.41091112992677,"color":"b"},{"x":246.47520988026736,"y":264.49216428506145,"color":"b"},{"x":251.71266154948017,"y":283.4827433391224,"color":"b"},{"x":194.21149447966374,"y":226.26467420764357,"color":"b"},{"x":213.26775768647832,"y":281.40519585164486,"color":"b"},{"x":242.77802625495752,"y":210.06309652482992,"color":"b"}]
df_draw = pd.DataFrame(data)
print("drawdata:", df_draw.shape, "| classes:", sorted(df_draw["color"].unique()))
df_draw.head()

On rejoue le scatter des données dessinées (les 4 classes colorées par la palette).

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
scatter_by_category(df_draw, "x", "y", "color", ax=ax,
                    title="drawdata — 4 classes dessinées à la main")
plt.show()

## 12. Données mixtes (num + cat) — Faker

Pour des données **tabulaires réalistes** (noms, e-mails, villes, dates), `sklearn` ne suffit pas. **[`Faker`](https://faker.readthedocs.io/)** génère des champs crédibles, localisables (`fr_FR`), reproductibles via `Faker.seed`. Idéal pour mock de pipeline, tests, démos sans PII réelle.

In [ ]:
from faker import Faker

fake = Faker("fr_FR")
Faker.seed(RANDOM_STATE)

df_people = pd.DataFrame([{
    "name": fake.name(),
    "email": fake.email(),
    "company": fake.company(),
    "city": fake.city(),
    "job": fake.job(),
    "birthdate": fake.date_of_birth(minimum_age=18, maximum_age=80),
    "salary": fake.random_int(20_000, 100_000),
} for _ in range(100)])
print("Faker:", df_people.shape)
df_people.head()

## 13. SDV & modèles génératifs (panorama 2026)

Quand on veut des données qui **conservent les corrélations** d'un vrai dataset (pour partager sans révéler la donnée), on passe aux **modèles génératifs tabulaires** :

- **[SDV](https://sdv.dev/) (Synthetic Data Vault)** — `GaussianCopulaSynthesizer` (rapide, corrélations linéaires), `CTGANSynthesizer` / `TVAESynthesizer` (deep, distributions complexes), + générateurs multi-tables et séries temporelles.
- **`ydata-synthetic`** — GAN-based.
- **Gretel.ai** — SaaS managé.

Esquisse d'API SDV (lib lourde, non installée dans cet env — schéma illustratif) :

```text
from sdv.metadata import SingleTableMetadata
from sdv.single_table import GaussianCopulaSynthesizer

metadata = SingleTableMetadata()
metadata.detect_from_dataframe(real_df)

synth = GaussianCopulaSynthesizer(metadata)
synth.fit(real_df)
fake_df = synth.sample(num_rows=1000)
```

⚠️ **Toujours évaluer** : (1) fidélité statistique (`sdmetrics`, comparaison de distributions / corrélations) et (2) confidentialité (risque de *membership inference* — un point synthétique trop proche d'un vrai point fuit de l'information).

## 14. Quand utiliser quoi

| Besoin | Outil |
|---|---|
| Illustrer une frontière 2D non-linéaire | `make_moons`, `make_circles`, `make_gaussian_quantiles` |
| Clusters convexes (tester KMeans/GMM) | `make_blobs` |
| Bench classifieur, contrôle fin des params | `make_classification` |
| Multi-label | `make_multilabel_classification` |
| Bench régression avec ground truth | `make_regression(coef=True)` |
| Tester la gestion du déséquilibre | `make_classification(weights=...)` / `imbalance_dataset` |
| Dataset large signal + bruit (sélection de variables) | composite from scratch (§9) |
| Cas pédagogique dessiné à la main | `drawdata` |
| Tabulaire réaliste (noms, dates, PII) | `Faker` |
| Reproduire un vrai dataset sans le révéler | `SDV` (GaussianCopula, CTGAN) |
| Séries temporelles synthétiques | voir `TS_Generer_Sequence` |

## 15. Sources

- [sklearn — Dataset generators](https://scikit-learn.org/stable/datasets/sample_generators.html)
- [drawdata — GitHub](https://github.com/koaning/drawdata)
- [Faker — docs](https://faker.readthedocs.io/)
- [SDV — Synthetic Data Vault](https://sdv.dev/)
- Notebooks liés : `TS_Generer_Sequence`, `NLP_Classification_Smote`, `ML_Regression_Classification_Multiple`, `ML_Explication_Feature_Importance_Selection`.